# 08-01 v2. Transfermarkt Data Integration
## Leakage-safe Preseason Player Snapshot 구축

### v2에서 수정한 핵심 문제

v1 데이터 품질 검사에서 다음 문제가 확인되었습니다.

1. Transfermarkt의 `Laliga`와 기존 데이터의 `La Liga` 표기가 달라 일부 Big 5 판정이 틀림
2. eordo summer event → DuckDB transfer date 연결률이 낮음
3. 같은 여름 여러 transfer event가 있을 때 하나만 선택해 방향이 뒤집힐 수 있음
4. `현재 팀 == to_club`인 임대→완전이적 같은 이벤트를 실제 팀 변경으로 잘못 해석할 수 있음
5. 새 팀 직전 시즌 strength coverage가 58.46%로 낮음
6. 날짜를 확인하지 못한 summer event를 실제 '이적 없음'과 구분해서 audit할 필요가 있음

v2에서는 **Player ID / Market Value 로직은 최대한 유지**하고,
Transfer / Club / New-team 부분을 집중적으로 수정합니다.

---

## 예측 시점

> 현재 시즌 T가 종료된 뒤, 다음 시즌 T+1의 Big 5 리그 중
> 가장 먼저 시작하는 리그의 개막 전날을 공통 prediction cutoff로 사용한다.

따라서:

- cutoff 이전에 날짜가 확인된 이적만 feature로 사용
- cutoff 이후 이적은 audit only
- exact date를 확인하지 못한 summer event도 audit only
- 미래 이적 존재 여부를 model feature로 사용하지 않음

---

## v2 주요 산출물

`artifacts/` 아래에 기존 v1과 구분되도록 `_v2` 이름으로 저장합니다.

## 0. 왜 08-01을 모델링과 분리하는가?

이번 단계에는 일반적인 feature engineering보다 훨씬 많은 데이터 문제가 있습니다.

- 서로 다른 데이터셋의 선수 이름 연결
- 동일 이름의 다른 선수 구분
- 클럽 이름 표기 차이
- 이적 이벤트의 중복(in/out) 제거
- 이적 시점 검증
- 역사적 시장가치의 point-in-time join
- 다음 시즌 성적을 실수로 붙이는 temporal leakage 방지
- 옛 시즌 시장가치 결측 문제

따라서 먼저 **신뢰할 수 있는 데이터셋**을 만든 뒤,
08-02에서 고정된 모델로 feature group ablation을 수행합니다.

In [2]:
from pathlib import Path
from collections import defaultdict
from difflib import SequenceMatcher
import random
import re
import unicodedata
import warnings
import zipfile

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ------------------------------------------------------------------
# 정책 설정
# ------------------------------------------------------------------

ALLOW_FUZZY_PLAYER_AUTO_MATCH = False

FALLBACK_CUTOFF_MONTH = 7
FALLBACK_CUTOFF_DAY = 31

MIN_PLAYER_ROW_MATCH_RATE = 0.80
MIN_OLD_TEAM_MATCH_RATE = 0.98

# Transfer event ↔ DB transfer date pair matching
TRANSFER_PAIR_MIN_SCORE = 0.60
TRANSFER_PAIR_STRONG_SCORE = 0.78

# 현재 팀 상태 ↔ transfer event의 from/to club 연결 기준
TEAM_CONTEXT_MIN_SCORE = 0.66

# final destination이 현재 팀과 같은 팀인지 판단
SAME_CLUB_SCORE = 0.82

# 새 팀 ↔ team_season_stats match
NEW_TEAM_MIN_SCORE = 0.64
NEW_TEAM_MIN_MARGIN = 0.04

MANUAL_PLAYER_ID_OVERRIDES = {
    # "Example Player": 123456,
}

MANUAL_DESTINATION_TEAM_OVERRIDES = {
    # "Paris Saint-Germain FC": "Paris S-G",
}

BIG5_LEAGUES = [
    "Premier League",
    "La Liga",
    "Bundesliga",
    "Serie A",
    "Ligue 1",
]

BIG5_COMPETITION_IDS = {
    "Premier League": "GB1",
    "La Liga": "ES1",
    "Bundesliga": "L1",
    "Serie A": "IT1",
    "Ligue 1": "FR1",
}

COMPETITION_ID_TO_BIG5_LEAGUE = {
    v: k for k, v in BIG5_COMPETITION_IDS.items()
}

LEAGUE_COUNTRY = {
    "Premier League": "England",
    "La Liga": "Spain",
    "Bundesliga": "Germany",
    "Serie A": "Italy",
    "Ligue 1": "France",
}

ZIP_FOLDER_TO_LEAGUE = {
    "premier_league": "Premier League",
    "laliga": "La Liga",
    "bundesliga": "Bundesliga",
    "serie_a": "Serie A",
    "ligue_1": "Ligue 1",
}

# eordo / Transfermarkt의 리그 표기 차이를 한 번에 통일
LEAGUE_ALIASES = {
    "premier league": "Premier League",
    "premierleague": "Premier League",
    "england premier league": "Premier League",

    "la liga": "La Liga",
    "laliga": "La Liga",
    "la liga ea sports": "La Liga",

    "bundesliga": "Bundesliga",
    "1 bundesliga": "Bundesliga",

    "serie a": "Serie A",
    "seriea": "Serie A",

    "ligue 1": "Ligue 1",
    "ligue1": "Ligue 1",
}

# Club 이름 비교 시 의미가 약한 일반 토큰
GENERIC_CLUB_TOKENS = {
    "fc", "cf", "ac", "afc", "sc", "ss", "bc", "sv", "vfl", "vfb",
    "club", "football", "futbol", "calcio", "de", "the"
}

# 자주 등장하는 축약 표기
CLUB_NAME_ALIASES = {
    "manchester utd": "manchester united",
    "man utd": "manchester united",
    "manchester city": "manchester city",
    "man city": "manchester city",
    "paris s g": "paris saint germain",
    "paris sg": "paris saint germain",
    "psg": "paris saint germain",
    "eint frankfurt": "eintracht frankfurt",
    "ein frankfurt": "eintracht frankfurt",
    "nott ham forest": "nottingham forest",
    "nottm forest": "nottingham forest",
    "dortmund": "borussia dortmund",
    "gladbach": "borussia monchengladbach",
    "m gladbach": "borussia monchengladbach",
    "leverkusen": "bayer leverkusen",
    "bayern munich": "bayern munich",
}

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("08-01 v2 settings loaded.")

08-01 v2 settings loaded.


## 1. DuckDB 패키지 확인

받은 `transfermarkt-datasets.duckdb`를 읽으려면 Python `duckdb` 패키지가 필요합니다.

설치되어 있지 않다면 아래 메시지의 명령을 실행한 뒤
**커널을 재시작하고 Notebook을 처음부터 다시 실행**하세요.

In [3]:
try:
    import duckdb
    print("duckdb version:", duckdb.__version__)
except ImportError as exc:
    raise ImportError(
        "duckdb 패키지가 필요합니다. 터미널 또는 새 셀에서 "
        "`pip install duckdb` 실행 후 커널을 재시작하세요."
    ) from exc

duckdb version: 1.5.5


## 2. 파일 자동 탐색

필요 파일:

- long_basic Train
- long_basic Validation
- `transfermarkt-data-master.zip`
- `transfermarkt-datasets.duckdb`
- `team_season_stats_2000_2024.csv`
- `team_name_alias_map.csv`

**Test CSV는 찾지도, 읽지도 않습니다.**

In [4]:
MANUAL_TRAIN_PATH = None
MANUAL_VALIDATION_PATH = None
MANUAL_TRANSFER_ZIP_PATH = None
TRANSFER_DATA_PATH = Path("../data/transfermarkt-data-master")
if not TRANSFER_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Transfermarkt 폴더가 없습니다: {TRANSFER_DATA_PATH}"
    )
MANUAL_TRANSFER_DUCKDB_PATH = None
MANUAL_TEAM_STATS_PATH = None
MANUAL_TEAM_ALIAS_PATH = None

SEARCH_ROOTS = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
]

SEARCH_ROOTS = list(
    dict.fromkeys(
        p.resolve()
        for p in SEARCH_ROOTS
        if p.exists()
    )
)

LONG_BASIC_REQUIRED = {
    "player", "nation", "team", "league", "season", "target_season",
    "position_group", "age", "matched_next", "starts", "minutes",
    "goals", "assists", "non_penalty_goals", "penalty_goals",
    "penalty_attempts", "goals_per90", "assists_per90",
    "goal_contrib_per90", "next_goals"
}

ADVANCED_FORBIDDEN = {
    "xg", "npxg", "xag", "shots", "progressive_carries"
}


def read_header(path):
    try:
        return set(pd.read_csv(path, nrows=0).columns)
    except Exception:
        return set()


def find_file_by_name_fragment(fragment, suffix=None):
    candidates = []

    for root in SEARCH_ROOTS:
        try:
            for path in root.rglob("*"):
                if not path.is_file():
                    continue
                if fragment.lower() not in path.name.lower():
                    continue
                if suffix is not None and path.suffix.lower() != suffix.lower():
                    continue
                candidates.append(path)
        except (PermissionError, OSError):
            continue

    if not candidates:
        return None

    candidates.sort(
        key=lambda p: (len(str(p)), str(p))
    )
    return candidates[0]


def find_long_basic_csv(kind):
    candidates = []

    for root in SEARCH_ROOTS:
        try:
            for path in root.rglob("*.csv"):
                cols = read_header(path)

                if not LONG_BASIC_REQUIRED.issubset(cols):
                    continue

                if ADVANCED_FORBIDDEN.intersection(cols):
                    continue

                name = path.name.lower()

                try:
                    sample = pd.read_csv(
                        path,
                        usecols=["season"],
                        nrows=200,
                    )
                    seasons = set(
                        sample["season"].astype(str)
                    )
                except Exception:
                    continue

                if kind == "train":
                    score = (
                        3 * int("train" in name)
                        + 3 * int("2000-2001" in seasons)
                        + int("validation" not in name)
                    )
                else:
                    score = (
                        3 * int(
                            "validation" in name
                            or "val" in name
                        )
                        + 3 * int("2023-2024" in seasons)
                        + int("train" not in name)
                    )

                candidates.append((score, path))
        except (PermissionError, OSError):
            continue

    if not candidates:
        return None

    candidates.sort(
        key=lambda x: (
            x[0],
            -len(str(x[1]))
        ),
        reverse=True,
    )

    return candidates[0][1]


def resolve_path(manual, auto):
    if manual is not None:
        path = Path(manual)
        if not path.exists():
            raise FileNotFoundError(
                f"직접 지정한 파일이 없습니다: {path}"
            )
        return path

    if auto is None:
        raise FileNotFoundError(
            "파일을 자동으로 찾지 못했습니다. "
            "상단 MANUAL_*_PATH에 직접 경로를 지정하세요."
        )

    return auto


TRAIN_PATH = resolve_path(
    MANUAL_TRAIN_PATH,
    find_long_basic_csv("train"),
)

VALIDATION_PATH = resolve_path(
    MANUAL_VALIDATION_PATH,
    find_long_basic_csv("validation"),
)


TRANSFER_DUCKDB_PATH = resolve_path(
    MANUAL_TRANSFER_DUCKDB_PATH,
    find_file_by_name_fragment(
        "transfermarkt-datasets",
        suffix=".duckdb",
    ),
)

TEAM_STATS_PATH = resolve_path(
    MANUAL_TEAM_STATS_PATH,
    find_file_by_name_fragment(
        "team_season_stats",
        suffix=".csv",
    ),
)

TEAM_ALIAS_PATH = resolve_path(
    MANUAL_TEAM_ALIAS_PATH,
    find_file_by_name_fragment(
        "team_name_alias",
        suffix=".csv",
    ),
)

print("TRAIN      :", TRAIN_PATH)
print("VALIDATION :", VALIDATION_PATH)
print("TM DATA    :", TRANSFER_DATA_PATH)
print("TM DUCKDB  :", TRANSFER_DUCKDB_PATH)
print("TEAM STATS :", TEAM_STATS_PATH)
print("TEAM ALIAS :", TEAM_ALIAS_PATH)

TRAIN      : D:\dev\03_PersonalProjects\next_season_goal_prediction\data\long_basic\train.csv
VALIDATION : D:\dev\03_PersonalProjects\next_season_goal_prediction\data\long_basic\validation.csv
TM DATA    : ..\data\transfermarkt-data-master
TM DUCKDB  : D:\dev\03_PersonalProjects\next_season_goal_prediction\data\transfermarkt-datasets.duckdb
TEAM STATS : D:\dev\03_PersonalProjects\next_season_goal_prediction\data\team_season_stats_2000_2024.csv
TEAM ALIAS : D:\dev\03_PersonalProjects\next_season_goal_prediction\data\team_name_alias_map.csv


## 3. 기존 개발 데이터 로드

Train + 기존 Validation만 합칩니다.

이 Notebook에서는 Final Test를 사용하지 않습니다.
또한 `matched_next=False`도 제거하지 않습니다.

이유:

- 08-02 Model B 회귀에서는 matched만 사용할 수 있음
- 09 Model A Big5 잔류/이탈 분류에서는 전체 행이 필요함

즉 외부 데이터셋은 **전체 모집단에 대해 한 번만 구축**합니다.

In [5]:
train_raw = pd.read_csv(TRAIN_PATH)
validation_raw = pd.read_csv(VALIDATION_PATH)

team_stats = pd.read_csv(TEAM_STATS_PATH)
team_alias = pd.read_csv(TEAM_ALIAS_PATH)

for df in [
    train_raw,
    validation_raw,
    team_stats,
    team_alias,
]:
    drop_cols = [
        c for c in df.columns
        if c.lower().startswith("unnamed")
        or c == "index"
    ]

    if drop_cols:
        df.drop(
            columns=drop_cols,
            inplace=True,
        )

dev = pd.concat(
    [train_raw, validation_raw],
    ignore_index=True,
).copy()

dev["season_start"] = (
    dev["season"]
    .astype(str)
    .str[:4]
    .astype(int)
)

dev["target_year"] = (
    dev["target_season"]
    .astype(str)
    .str[:4]
    .astype(int)
)

dev["row_id"] = np.arange(
    len(dev),
    dtype=int,
)

print("Train shape      :", train_raw.shape)
print("Validation shape :", validation_raw.shape)
print("Development shape:", dev.shape)

print(
    "Season range:",
    dev["season"].min(),
    "~",
    dev["season"].max(),
)

print(
    "matched_next distribution:"
)
display(
    dev["matched_next"]
    .value_counts(dropna=False)
    .rename("n")
    .to_frame()
)

Train shape      : (22430, 22)
Validation shape : (923, 22)
Development shape: (23353, 25)
Season range: 2000-2001 ~ 2023-2024
matched_next distribution:


,n
matched_next,
True,19403
False,3950


## 4. 문자열 정규화 함수

외부 데이터 연결에서 가장 흔한 문제는 표기 차이입니다.

예:

```text
Kylian Mbappé ↔ Kylian Mbappe
Paris Saint-Germain ↔ Paris S-G
1. FC Köln ↔ Köln
```

선수 이름은 악센트·문장부호를 제거한 정규화 키를 만듭니다.

클럽은 지나치게 공격적으로 단어를 삭제하면 서로 다른 클럽이 합쳐질 수 있으므로
기본 정규화 + fuzzy similarity를 함께 사용합니다.

In [6]:
def normalize_text(value):
    if pd.isna(value):
        return ""

    text = str(value).strip().lower()

    text = unicodedata.normalize(
        "NFKD",
        text,
    )

    text = "".join(
        ch
        for ch in text
        if not unicodedata.combining(ch)
    )

    replacements = {
        "ß": "ss",
        "ø": "o",
        "đ": "d",
        "ł": "l",
        "ð": "d",
        "þ": "th",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    text = re.sub(
        r"[^a-z0-9]+",
        " ",
        text,
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text


def canonicalize_league(value):
    if pd.isna(value):
        return np.nan

    key = normalize_text(value)

    return LEAGUE_ALIASES.get(
        key,
        str(value).strip(),
    )


def normalize_club(value):
    text = normalize_text(value)

    if not text:
        return ""

    text = CLUB_NAME_ALIASES.get(
        text,
        text,
    )

    return text


def club_core_tokens(value):
    text = normalize_club(value)

    tokens = [
        token
        for token in text.split()
        if token not in GENERIC_CLUB_TOKENS
        and not token.isdigit()
    ]

    return tokens


def string_similarity(a, b):
    a = normalize_text(a)
    b = normalize_text(b)

    if not a or not b:
        return 0.0

    if a == b:
        return 1.0

    return SequenceMatcher(
        None,
        a,
        b,
    ).ratio()


def club_similarity(a, b):
    """
    일반 문자열 similarity보다 club 표기에 강한 비교 함수.

    예:
    - Getafe CF ↔ Getafe
    - Olympique Marseille ↔ Marseille
    - Borussia Dortmund ↔ Dortmund
    """
    a_norm = normalize_club(a)
    b_norm = normalize_club(b)

    if not a_norm or not b_norm:
        return 0.0

    if a_norm == b_norm:
        return 1.0

    base = SequenceMatcher(
        None,
        a_norm,
        b_norm,
    ).ratio()

    a_tokens = club_core_tokens(a)
    b_tokens = club_core_tokens(b)

    if not a_tokens or not b_tokens:
        return base

    a_set = set(a_tokens)
    b_set = set(b_tokens)

    # FC / CF / AC 같은 일반 토큰 제거 후 동일
    if a_set == b_set:
        return max(base, 0.98)

    # Marseille ↔ Olympique Marseille,
    # Dortmund ↔ Borussia Dortmund 같은 관계
    smaller = (
        a_set if len(a_set) <= len(b_set)
        else b_set
    )
    larger = (
        b_set if len(a_set) <= len(b_set)
        else a_set
    )

    distinctive = [
        t
        for t in smaller
        if len(t) >= 5
    ]

    if (
        distinctive
        and smaller.issubset(larger)
    ):
        base = max(base, 0.92)

    return float(base)


dev["norm_player"] = (
    dev["player"]
    .map(normalize_text)
)

dev["norm_current_team"] = (
    dev["team"]
    .map(normalize_club)
)

print(
    dev[
        [
            "player",
            "norm_player",
            "team",
            "norm_current_team",
        ]
    ].head()
)

                player          norm_player            team norm_current_team
0  Abdelhafid Tasfaout  abdelhafid tasfaout        Guingamp          guingamp
1        Abder Ramdane        abder ramdane        Freiburg          freiburg
2             Adaílton             adailton   Hellas Verona     hellas verona
3         Ade Akinbiyi         ade akinbiyi  Leicester City    leicester city
4         Adel Sellimi         adel sellimi        Freiburg          freiburg


# Part A. Transfermarkt 원본 데이터 확인

## 5. eordo Transfermarkt ZIP 로드

받은 ZIP에서 Big 5의 2000~2025 시즌 CSV를 직접 읽습니다.

이 데이터의 주요 장점:

- `player_id`
- summer / winter
- in / out
- 이적 당시 시장가치
- dealing club / country
- fee
- loan

ZIP의 `season=2024`는 **2024-25 시즌**을 의미합니다.

In [7]:
def load_eordo_big5_folder(
    root_path,
    start_year=2000,
    end_year=2025,
):
    frames = []

    root_path = Path(root_path)

    for folder, expected_league in ZIP_FOLDER_TO_LEAGUE.items():
        league_dir = root_path / folder

        if not league_dir.exists():
            print(
                f"[WARNING] 폴더 없음: {league_dir}"
            )
            continue

        for year in range(
            start_year,
            end_year + 1,
        ):
            csv_path = (
                league_dir
                / f"{year}.csv"
            )

            if not csv_path.exists():
                continue

            tmp = pd.read_csv(
                csv_path
            )

            tmp[
                "source_folder"
            ] = folder

            tmp[
                "source_file"
            ] = str(csv_path)

            frames.append(tmp)

    if not frames:
        raise ValueError(
            "Big5 Transfermarkt CSV를 찾지 못했습니다."
        )

    return pd.concat(
        frames,
        ignore_index=True,
    )


eordo = load_eordo_big5_folder(
    TRANSFER_DATA_PATH,
    start_year=2000,
    end_year=2025,
)

eordo["player_id"] = pd.to_numeric(
    eordo["player_id"],
    errors="coerce",
).astype("Int64")

eordo["season"] = pd.to_numeric(
    eordo["season"],
    errors="coerce",
).astype("Int64")

eordo["norm_player"] = (
    eordo["player_name"]
    .map(normalize_text)
)

# v2 핵심: Laliga → La Liga 등 표기 통일
eordo[
    "league_raw"
] = eordo["league"]

eordo[
    "league"
] = (
    eordo["league"]
    .map(canonicalize_league)
)

league_normalization_changes = (
    eordo["league_raw"]
    .astype(str)
    .ne(
        eordo["league"]
        .astype(str)
    )
    .sum()
)

print(
    "Rows:",
    len(eordo),
)

print(
    "Unique player IDs:",
    eordo["player_id"].nunique(),
)

print(
    "Season:",
    eordo["season"].min(),
    "~",
    eordo["season"].max(),
)

print(
    "League labels normalized:",
    int(league_normalization_changes),
)

print(
    "Canonical leagues:",
    sorted(
        eordo["league"]
        .dropna()
        .astype(str)
        .unique()
    )[:20],
)

display(
    eordo[
        [
            "season",
            "league_raw",
            "league",
            "club",
            "window",
            "movement",
            "player_name",
            "player_id",
            "market_value",
            "dealing_club",
            "fee",
            "is_loan",
        ]
    ].head()
)

Rows: 78460
Unique player IDs: 21154
Season: 2000 ~ 2025
League labels normalized: 12992
Canonical leagues: ['Bundesliga', 'La Liga', 'Ligue 1', 'Premier League', 'Serie A']


,season,league_raw,league,club,window,movement,player_name,player_id,market_value,dealing_club,fee,is_loan
0,2000,Premier League,Premier League,Arsenal FC,summer,in,Sylvain Wiltord,3188,NaN,FC Girondins Bordeaux,17500000.0,0
1,2000,Premier League,Premier League,Arsenal FC,summer,in,Francis Jeffers,3186,NaN,Everton FC,15300000.0,0
2,2000,Premier League,Premier League,Arsenal FC,summer,in,Laurén,3189,NaN,RCD Mallorca,10700000.0,0
3,2000,Premier League,Premier League,Arsenal FC,summer,in,Robert Pirès,3185,NaN,Olympique Marseille,9800000.0,0
4,2000,Premier League,Premier League,Arsenal FC,summer,in,Igors Stepanovs,3200,NaN,Skonto Riga (- 2016),1500000.0,0


## 6. eordo 데이터 기본 품질 확인

In [8]:
eordo_quality = pd.DataFrame({
    "metric": [
        "all_rows",
        "unique_player_ids",
        "summer_rows",
        "summer_in_rows",
        "summer_out_rows",
        "market_value_nonnull_rate_summer",
        "fee_nonnull_rate_summer",
    ],
    "value": [
        len(eordo),
        eordo["player_id"].nunique(),
        (
            eordo["window"]
            .astype(str)
            .str.lower()
            .eq("summer")
            .sum()
        ),
        (
            eordo["window"]
            .astype(str)
            .str.lower()
            .eq("summer")
            & eordo["movement"]
            .astype(str)
            .str.lower()
            .eq("in")
        ).sum(),
        (
            eordo["window"]
            .astype(str)
            .str.lower()
            .eq("summer")
            & eordo["movement"]
            .astype(str)
            .str.lower()
            .eq("out")
        ).sum(),
        eordo.loc[
            eordo["window"]
            .astype(str)
            .str.lower()
            .eq("summer"),
            "market_value",
        ].notna().mean(),
        eordo.loc[
            eordo["window"]
            .astype(str)
            .str.lower()
            .eq("summer"),
            "fee",
        ].notna().mean(),
    ],
})

eordo_quality

,metric,value
0,all_rows,78460.000000
1,unique_player_ids,21154.000000
2,summer_rows,60759.000000
3,summer_in_rows,20924.000000
4,summer_out_rows,39835.000000
5,market_value_nonnull_rate_summer,0.800556
6,fee_nonnull_rate_summer,0.881285


## 7. DuckDB 테이블과 스키마 확인

`transfermarkt-datasets.duckdb`는:

- players
- player_valuations
- transfers
- games

를 주로 사용합니다.

여기서는 실제 파일의 column을 출력해서
나중에 데이터 버전이 바뀌더라도 바로 확인할 수 있게 합니다.

In [9]:
con = duckdb.connect(
    str(TRANSFER_DUCKDB_PATH),
    read_only=True,
)

tables = (
    con.execute("SHOW TABLES")
    .df()
)

display(tables)

REQUIRED_DUCKDB_TABLES = {
    "players",
    "player_valuations",
    "transfers",
    "games",
}

available_tables = set(
    tables.iloc[:, 0].astype(str)
)

missing_tables = (
    REQUIRED_DUCKDB_TABLES
    - available_tables
)

assert not missing_tables, (
    f"DuckDB 필수 테이블 누락: {missing_tables}"
)

for table in [
    "players",
    "player_valuations",
    "transfers",
    "games",
]:
    print("\n", "=" * 70)
    print(table)
    print("=" * 70)

    display(
        con.execute(
            f"DESCRIBE {table}"
        ).df()
    )

,name
0,appearances
1,club_games
2,clubs
3,competitions
4,countries
5,game_events
6,game_lineups
7,games
8,national_teams
9,player_valuations



players


,column_name,column_type,null,key,default,extra
0,player_id,INTEGER,YES,None,None,None
1,first_name,VARCHAR,YES,None,None,None
2,last_name,VARCHAR,YES,None,None,None
3,name,VARCHAR,YES,None,None,None
4,last_season,VARCHAR,YES,None,None,None
5,current_club_id,VARCHAR,YES,None,None,None
6,player_code,VARCHAR,YES,None,None,None
7,country_of_birth,VARCHAR,YES,None,None,None
8,city_of_birth,VARCHAR,YES,None,None,None
9,country_of_citizenship,VARCHAR,YES,None,None,None



player_valuations


,column_name,column_type,null,key,default,extra
0,player_id,INTEGER,YES,None,None,None
1,date,DATE,YES,None,None,None
2,market_value_in_eur,INTEGER,YES,None,None,None
3,current_club_name,VARCHAR,YES,None,None,None
4,current_club_id,INTEGER,YES,None,None,None
5,player_club_domestic_competition_id,VARCHAR,YES,None,None,None



transfers


,column_name,column_type,null,key,default,extra
0,player_id,INTEGER,YES,None,None,None
1,transfer_date,DATE,YES,None,None,None
2,transfer_season,VARCHAR,YES,None,None,None
3,from_club_id,INTEGER,YES,None,None,None
4,to_club_id,INTEGER,YES,None,None,None
5,from_club_name,VARCHAR,YES,None,None,None
6,to_club_name,VARCHAR,YES,None,None,None
7,transfer_fee,"DECIMAL(18,3)",YES,None,None,None
8,market_value_in_eur,"DECIMAL(18,3)",YES,None,None,None
9,player_name,VARCHAR,YES,None,None,None



games


,column_name,column_type,null,key,default,extra
0,game_id,VARCHAR,YES,None,None,None
1,competition_id,VARCHAR,YES,None,None,None
2,season,VARCHAR,YES,None,None,None
3,round,VARCHAR,YES,None,None,None
4,date,DATE,YES,None,None,None
5,home_club_id,INTEGER,YES,None,None,None
6,away_club_id,INTEGER,YES,None,None,None
7,home_club_goals,INTEGER,YES,None,None,None
8,away_club_goals,INTEGER,YES,None,None,None
9,home_club_position,INTEGER,YES,None,None,None


## 8. DuckDB 필요 테이블 로드

현재 파일 규모에서는 필요한 4개 테이블만 pandas로 읽어도 충분합니다.

`appearances`, `game_events` 같은 대용량 테이블은 이번 단계에 필요하지 않습니다.

In [10]:
tm_players_raw = (
    con.execute(
        "SELECT * FROM players"
    ).df()
)

valuations_raw = (
    con.execute(
        "SELECT * FROM player_valuations"
    ).df()
)

db_transfers_raw = (
    con.execute(
        "SELECT * FROM transfers"
    ).df()
)

games_raw = (
    con.execute(
        "SELECT * FROM games"
    ).df()
)

print(
    "players          :",
    tm_players_raw.shape,
)
print(
    "player_valuations:",
    valuations_raw.shape,
)
print(
    "transfers        :",
    db_transfers_raw.shape,
)
print(
    "games            :",
    games_raw.shape,
)

players          : (50149, 26)
player_valuations: (656301, 6)
transfers        : (175165, 10)
games            : (88958, 23)


## 9. Column resolver

Transfermarkt dataset 버전에 따라 날짜 column 등이 조금 달라질 가능성에 대비해서
후보 이름 중 실제 존재하는 column을 자동으로 선택합니다.

In [11]:
def resolve_col(
    df,
    candidates,
    required=True,
):
    for col in candidates:
        if col in df.columns:
            return col

    if required:
        raise KeyError(
            f"다음 후보 column을 찾지 못했습니다: {candidates}\n"
            f"실제 columns: {df.columns.tolist()}"
        )

    return None


PLAYER_ID_COL = resolve_col(
    tm_players_raw,
    ["player_id"],
)

PLAYER_NAME_COL = resolve_col(
    tm_players_raw,
    ["name", "player_name"],
)

PLAYER_DOB_COL = resolve_col(
    tm_players_raw,
    ["date_of_birth", "birth_date"],
    required=False,
)

PLAYER_COUNTRY_COL = resolve_col(
    tm_players_raw,
    [
        "country_of_citizenship",
        "nationality",
        "country",
    ],
    required=False,
)

VALUATION_PLAYER_ID_COL = resolve_col(
    valuations_raw,
    ["player_id"],
)

VALUATION_DATE_COL = resolve_col(
    valuations_raw,
    ["date", "datetime"],
)

VALUATION_VALUE_COL = resolve_col(
    valuations_raw,
    [
        "market_value_in_eur",
        "market_value",
    ],
)

TRANSFER_PLAYER_ID_COL = resolve_col(
    db_transfers_raw,
    ["player_id"],
)

TRANSFER_DATE_COL = resolve_col(
    db_transfers_raw,
    ["transfer_date", "date"],
)

TRANSFER_FROM_COL = resolve_col(
    db_transfers_raw,
    ["from_club_name", "from_club"],
)

TRANSFER_TO_COL = resolve_col(
    db_transfers_raw,
    ["to_club_name", "to_club"],
)

TRANSFER_FEE_COL = resolve_col(
    db_transfers_raw,
    ["transfer_fee", "fee"],
    required=False,
)

TRANSFER_MV_COL = resolve_col(
    db_transfers_raw,
    [
        "market_value_in_eur",
        "market_value",
    ],
    required=False,
)

GAME_COMP_COL = resolve_col(
    games_raw,
    ["competition_id"],
)

GAME_SEASON_COL = resolve_col(
    games_raw,
    ["season"],
)

GAME_DATE_COL = resolve_col(
    games_raw,
    ["date"],
)

print(
    "Resolved schema successfully."
)

Resolved schema successfully.


# Part B. Player ID Crosswalk

## 10. Transfermarkt 선수 master 생성

두 소스를 합쳐 `normalized name → player_id` 후보를 만듭니다.

- DuckDB `players`: 선수 master + 생년월일
- eordo ZIP: 이적 당시 이름 + player_id

### 자동 확정 원칙

**A 등급**
- 정규화 이름이 Transfermarkt 전체에서 단 하나의 player_id에만 연결됨

**B 등급**
- 동일 이름에 여러 player_id가 있으나 생년월일 기반 시즌 나이가 명확히 한 후보와 일치

**D 등급**
- fuzzy 후보
- 기본적으로 자동 확정하지 않고 audit만 생성

정확도를 coverage보다 우선합니다.

In [12]:
tm_players = pd.DataFrame({
    "player_id": pd.to_numeric(
        tm_players_raw[PLAYER_ID_COL],
        errors="coerce",
    ),
    "tm_player_name": (
        tm_players_raw[PLAYER_NAME_COL]
        .astype(str)
    ),
})

if PLAYER_DOB_COL is not None:
    tm_players["date_of_birth"] = pd.to_datetime(
        tm_players_raw[PLAYER_DOB_COL],
        errors="coerce",
    )
else:
    tm_players["date_of_birth"] = pd.NaT

if PLAYER_COUNTRY_COL is not None:
    tm_players["country_of_citizenship"] = (
        tm_players_raw[PLAYER_COUNTRY_COL]
        .astype(str)
    )
else:
    tm_players["country_of_citizenship"] = np.nan

tm_players = tm_players.dropna(
    subset=["player_id"]
).copy()

tm_players["player_id"] = (
    tm_players["player_id"]
    .astype(int)
)

tm_players["norm_player"] = (
    tm_players["tm_player_name"]
    .map(normalize_text)
)

eordo_name_ids = (
    eordo[
        [
            "player_id",
            "player_name",
            "norm_player",
        ]
    ]
    .dropna(subset=["player_id"])
    .drop_duplicates()
    .rename(
        columns={
            "player_name": "tm_player_name"
        }
    )
)

eordo_name_ids["player_id"] = (
    eordo_name_ids["player_id"]
    .astype(int)
)

duckdb_name_ids = (
    tm_players[
        [
            "player_id",
            "tm_player_name",
            "norm_player",
        ]
    ]
    .drop_duplicates()
)

name_id_master = pd.concat(
    [
        duckdb_name_ids,
        eordo_name_ids,
    ],
    ignore_index=True,
).drop_duplicates()

name_id_counts = (
    name_id_master
    .groupby("norm_player")
    ["player_id"]
    .nunique()
    .rename("candidate_id_count")
)

print(
    "Normalized Transfermarkt names:",
    name_id_counts.shape[0],
)

print(
    "Ambiguous normalized names:",
    int(
        (name_id_counts > 1).sum()
    ),
)

Normalized Transfermarkt names: 57713
Ambiguous normalized names: 1249


## 11. A 등급: Exact normalized name + unique player_id

In [13]:
player_crosswalk = (
    dev[
        ["player", "norm_player"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

candidate_ids_by_name = (
    name_id_master
    .groupby("norm_player")
    ["player_id"]
    .agg(
        lambda s: sorted(
            set(
                int(x)
                for x in s
                if pd.notna(x)
            )
        )
    )
)

player_crosswalk[
    "candidate_ids"
] = (
    player_crosswalk[
        "norm_player"
    ]
    .map(candidate_ids_by_name)
)

player_crosswalk[
    "candidate_ids"
] = (
    player_crosswalk[
        "candidate_ids"
    ]
    .apply(
        lambda x: (
            x
            if isinstance(x, list)
            else []
        )
    )
)

player_crosswalk[
    "candidate_id_count"
] = (
    player_crosswalk[
        "candidate_ids"
    ].map(len)
)

player_crosswalk[
    "tm_player_id"
] = np.nan

player_crosswalk[
    "match_method"
] = "unmatched"

player_crosswalk[
    "match_confidence"
] = "UNMATCHED"

mask_a = (
    player_crosswalk[
        "candidate_id_count"
    ]
    .eq(1)
)

player_crosswalk.loc[
    mask_a,
    "tm_player_id",
] = (
    player_crosswalk.loc[
        mask_a,
        "candidate_ids",
    ]
    .map(lambda x: x[0])
)

player_crosswalk.loc[
    mask_a,
    "match_method",
] = "exact_normalized_unique"

player_crosswalk.loc[
    mask_a,
    "match_confidence",
] = "A"

print(
    "A-grade players:",
    int(mask_a.sum()),
    "/",
    len(player_crosswalk),
    f"({mask_a.mean():.2%})",
)

A-grade players: 5374 / 6161 (87.23%)


## 12. B 등급: 동일 이름 여러 선수 → 생년월일/나이로 해결

예를 들어 `Fernando`, `Eduardo`처럼 동일 이름을 가진 여러 player_id가 존재할 수 있습니다.

각 후보의 생년월일로 현재 시즌 7월 1일 기준 예상 나이를 계산하고,
우리 데이터의 여러 시즌 나이와 가장 일관되게 맞는 ID를 찾습니다.

### B 등급 자동 확정 조건

- best 후보 median age error ≤ 1년
- 두 번째 후보보다 충분히 명확하게 좋음

조건이 애매하면 자동 확정하지 않습니다.

In [14]:
dob_by_id = (
    tm_players
    .drop_duplicates(
        subset=["player_id"]
    )
    .set_index("player_id")
    ["date_of_birth"]
    .to_dict()
)


def expected_age_on_july1(
    season_start,
    dob,
):
    if pd.isna(dob):
        return np.nan

    ref = pd.Timestamp(
        year=int(season_start),
        month=7,
        day=1,
    )

    return (
        (ref - pd.Timestamp(dob)).days
        / 365.2425
    )


def resolve_ambiguous_by_age(
    norm_player,
    candidate_ids,
):
    obs = dev.loc[
        dev["norm_player"].eq(
            norm_player
        ),
        ["season_start", "age"],
    ].dropna()

    if obs.empty:
        return None

    scores = []

    for pid in candidate_ids:
        dob = dob_by_id.get(
            int(pid),
            pd.NaT,
        )

        if pd.isna(dob):
            continue

        diffs = []

        for row in obs.itertuples(
            index=False
        ):
            exp_age = expected_age_on_july1(
                row.season_start,
                dob,
            )

            if pd.isna(exp_age):
                continue

            diffs.append(
                abs(
                    float(row.age)
                    - exp_age
                )
            )

        if diffs:
            scores.append({
                "player_id": int(pid),
                "median_age_error": float(
                    np.median(diffs)
                ),
                "mean_age_error": float(
                    np.mean(diffs)
                ),
                "n_age_rows": len(diffs),
            })

    if not scores:
        return None

    scores = sorted(
        scores,
        key=lambda x: (
            x["median_age_error"],
            x["mean_age_error"],
            -x["n_age_rows"],
        ),
    )

    best = scores[0]

    second_error = (
        scores[1]["median_age_error"]
        if len(scores) > 1
        else np.inf
    )

    # 나이 기준이 충분히 맞고
    # 2위와 구분될 때만 자동 확정
    if (
        best["median_age_error"] <= 1.0
        and (
            second_error
            - best["median_age_error"]
        ) >= 0.5
    ):
        return best

    return None


ambiguous_mask = (
    player_crosswalk[
        "candidate_id_count"
    ] > 1
)

resolved_b = 0

for idx in player_crosswalk.loc[
    ambiguous_mask
].index:

    norm_name = player_crosswalk.at[
        idx,
        "norm_player",
    ]

    candidates = player_crosswalk.at[
        idx,
        "candidate_ids",
    ]

    result = resolve_ambiguous_by_age(
        norm_name,
        candidates,
    )

    if result is None:
        continue

    player_crosswalk.at[
        idx,
        "tm_player_id",
    ] = result["player_id"]

    player_crosswalk.at[
        idx,
        "match_method",
    ] = "exact_name_age_resolved"

    player_crosswalk.at[
        idx,
        "match_confidence",
    ] = "B"

    player_crosswalk.at[
        idx,
        "age_match_error",
    ] = result[
        "median_age_error"
    ]

    resolved_b += 1


print(
    "B-grade resolved:",
    resolved_b,
)

B-grade resolved: 178


## 13. D 등급 fuzzy 후보 생성 — Audit Only

Exact normalized name으로 연결되지 않은 선수에 대해서는
가장 비슷한 Transfermarkt 이름 3개를 저장합니다.

기본 설정에서는 **fuzzy 후보를 자동으로 player_id에 넣지 않습니다.**

이 파일을 보고 정말 필요한 선수를 수동 검토한 뒤
`MANUAL_PLAYER_ID_OVERRIDES`에 추가하는 방식이 안전합니다.

In [15]:
tm_norm_names = sorted(
    name_id_master[
        "norm_player"
    ]
    .dropna()
    .unique()
)

unmatched_names = (
    player_crosswalk.loc[
        player_crosswalk[
            "tm_player_id"
        ].isna(),
        [
            "player",
            "norm_player",
        ],
    ]
    .drop_duplicates()
)

try:
    from rapidfuzz import process, fuzz

    FUZZY_ENGINE = "rapidfuzz"

    fuzzy_rows = []

    for row in unmatched_names.itertuples(
        index=False
    ):
        matches = process.extract(
            row.norm_player,
            tm_norm_names,
            scorer=fuzz.WRatio,
            limit=3,
        )

        for rank, (
            candidate_name,
            score,
            _,
        ) in enumerate(
            matches,
            start=1,
        ):
            ids = (
                name_id_master.loc[
                    name_id_master[
                        "norm_player"
                    ].eq(candidate_name),
                    "player_id",
                ]
                .dropna()
                .astype(int)
                .unique()
                .tolist()
            )

            fuzzy_rows.append({
                "player": row.player,
                "norm_player": row.norm_player,
                "rank": rank,
                "candidate_norm_name": candidate_name,
                "similarity": score / 100.0,
                "candidate_player_ids": ids,
            })

except ImportError:
    import difflib

    FUZZY_ENGINE = "difflib"

    fuzzy_rows = []

    for row in unmatched_names.itertuples(
        index=False
    ):
        matches = difflib.get_close_matches(
            row.norm_player,
            tm_norm_names,
            n=3,
            cutoff=0.70,
        )

        for rank, candidate_name in enumerate(
            matches,
            start=1,
        ):
            score = string_similarity(
                row.norm_player,
                candidate_name,
            )

            ids = (
                name_id_master.loc[
                    name_id_master[
                        "norm_player"
                    ].eq(candidate_name),
                    "player_id",
                ]
                .dropna()
                .astype(int)
                .unique()
                .tolist()
            )

            fuzzy_rows.append({
                "player": row.player,
                "norm_player": row.norm_player,
                "rank": rank,
                "candidate_norm_name": candidate_name,
                "similarity": score,
                "candidate_player_ids": ids,
            })


player_match_audit = pd.DataFrame(
    fuzzy_rows
)

print(
    "Fuzzy engine:",
    FUZZY_ENGINE,
)

print(
    "Unmatched unique players:",
    len(unmatched_names),
)

display(
    player_match_audit.head(20)
)

Fuzzy engine: difflib
Unmatched unique players: 609


,player,norm_player,rank,candidate_norm_name,similarity,candidate_player_ids
0,Abder Ramdane,abder ramdane,1,ylber ramadani,0.740741,[442703]
1,Abder Ramdane,abder ramdane,2,abderrahmane sarr,0.733333,[1144153]
2,Abder Ramdane,abder ramdane,3,abde raihani,0.720000,[860062]
3,Adaílton,adailton,1,adailton,1.000000,"[101253, 21853, 18699, 34371]"
4,Adaílton,adailton,2,ailton,0.857143,"[32860, 283863, 353214, 516]"
5,Adaílton,adailton,3,mailton,0.800000,[607210]
6,Adel Sellimi,adel sellimi,1,adul seidi,0.727273,[316828]
7,Aílton Gonçalves,ailton goncalves,1,anthony goncalves,0.848485,[111528]
8,Aílton Gonçalves,ailton goncalves,2,vitor goncalves,0.838710,[194547]
9,Aílton Gonçalves,ailton goncalves,3,ivo goncalves,0.827586,[45243]


## 14. 수동 player_id override 적용

Fuzzy audit를 검토한 뒤 확실한 선수만 위의
`MANUAL_PLAYER_ID_OVERRIDES`에 추가하면 됩니다.

Notebook을 다시 실행하면 자동 적용됩니다.

In [16]:
override_map = {
    normalize_text(name): int(pid)
    for name, pid
    in MANUAL_PLAYER_ID_OVERRIDES.items()
}

manual_mask = (
    player_crosswalk[
        "norm_player"
    ].isin(
        override_map.keys()
    )
)

if manual_mask.any():
    player_crosswalk.loc[
        manual_mask,
        "tm_player_id",
    ] = (
        player_crosswalk.loc[
            manual_mask,
            "norm_player",
        ]
        .map(override_map)
    )

    player_crosswalk.loc[
        manual_mask,
        "match_method",
    ] = "manual_override"

    player_crosswalk.loc[
        manual_mask,
        "match_confidence",
    ] = "MANUAL"


player_crosswalk[
    "tm_player_id"
] = (
    pd.to_numeric(
        player_crosswalk[
            "tm_player_id"
        ],
        errors="coerce",
    )
    .astype("Int64")
)

display(
    player_crosswalk[
        "match_confidence"
    ]
    .value_counts(
        dropna=False
    )
    .rename("n")
    .to_frame()
)

,n
match_confidence,
A,5374
UNMATCHED,609
B,178


## 15. 선수 ID를 모든 player-season 행에 연결

In [17]:
snapshot = dev.merge(
    player_crosswalk[
        [
            "player",
            "norm_player",
            "tm_player_id",
            "match_method",
            "match_confidence",
        ]
    ],
    on=[
        "player",
        "norm_player",
    ],
    how="left",
    validate="many_to_one",
)

player_row_match_rate = (
    snapshot[
        "tm_player_id"
    ]
    .notna()
    .mean()
)

player_unique_match_rate = (
    player_crosswalk[
        "tm_player_id"
    ]
    .notna()
    .mean()
)

print(
    f"Unique player ID match: "
    f"{player_unique_match_rate:.2%}"
)

print(
    f"Player-season row match: "
    f"{player_row_match_rate:.2%}"
)

assert (
    player_row_match_rate
    >= MIN_PLAYER_ROW_MATCH_RATE
), (
    "Player ID row 매칭률이 예상보다 낮습니다. "
    "crosswalk audit를 먼저 확인하세요."
)

player_season_crosswalk = (
    snapshot[
        [
            "row_id",
            "player",
            "season",
            "team",
            "age",
            "tm_player_id",
            "match_method",
            "match_confidence",
        ]
    ]
    .copy()
)

unmatched_players = (
    player_crosswalk.loc[
        player_crosswalk[
            "tm_player_id"
        ].isna()
    ]
    .copy()
)

display(
    unmatched_players.head(20)
)

Unique player ID match: 90.12%
Player-season row match: 92.51%


,player,norm_player,candidate_ids,candidate_id_count,tm_player_id,match_method,match_confidence,age_match_error
1,Abder Ramdane,abder ramdane,[],0,<NA>,unmatched,UNMATCHED,NaN
2,Adaílton,adailton,"[18699, 21853, 34371, 101253]",4,<NA>,unmatched,UNMATCHED,NaN
4,Adel Sellimi,adel sellimi,[],0,<NA>,unmatched,UNMATCHED,NaN
8,Aílton Gonçalves,ailton goncalves,[],0,<NA>,unmatched,UNMATCHED,NaN
17,Aleksandr Mostovoi,aleksandr mostovoi,[],0,<NA>,unmatched,UNMATCHED,NaN
18,Aleksandre Iashvili,aleksandre iashvili,[],0,<NA>,unmatched,UNMATCHED,NaN
22,Alessandro Mazzola,alessandro mazzola,[],0,<NA>,unmatched,UNMATCHED,NaN
26,Alex Fernández,alex fernandez,"[7885, 89733]",2,<NA>,unmatched,UNMATCHED,NaN
30,Alexander Strehmel,alexander strehmel,[],0,<NA>,unmatched,UNMATCHED,NaN
32,Alexey Smertin,alexey smertin,[],0,<NA>,unmatched,UNMATCHED,NaN


# Part C. Prediction Cutoff

## 16. 시즌별 cutoff 생성

### 기본 정책

가능한 시즌:
- DuckDB `games`에서 Big 5 리그 각각의 첫 경기 날짜를 찾음
- **Big 5 전체 중 가장 빠른 개막일의 전날**을 cutoff로 사용

왜 한 리그별 cutoff가 아니라 Big 5 공통 cutoff인가?

예를 들어 EPL이 이미 시작된 뒤 Bundesliga 개막 직전 정보를 사용하면
Bundesliga 선수는 더 많은 미래 정보를 갖게 됩니다.

따라서 같은 시즌 모든 선수에게 하나의 timestamp를 적용합니다.

### 과거 경기 일정이 DuckDB에 없는 시즌

`YYYY-07-31`을 conservative fallback으로 사용하고
`cutoff_source = fallback_jul31`로 명확히 표시합니다.

08-02에서 필요하면 fallback 시즌만 제외한 sensitivity analysis도 할 수 있습니다.

In [18]:
games = games_raw.copy()

games["competition_id"] = (
    games[GAME_COMP_COL]
    .astype(str)
)

games["season_start"] = (
    pd.to_numeric(
        games[GAME_SEASON_COL],
        errors="coerce",
    )
)

games["game_date"] = (
    pd.to_datetime(
        games[GAME_DATE_COL],
        errors="coerce",
    )
)

big5_game_rows = (
    games[
        games["competition_id"]
        .isin(
            BIG5_COMPETITION_IDS.values()
        )
    ]
    .dropna(
        subset=[
            "season_start",
            "game_date",
        ]
    )
    .copy()
)

big5_game_rows[
    "season_start"
] = (
    big5_game_rows[
        "season_start"
    ]
    .astype(int)
)

first_big5_game_by_year = (
    big5_game_rows
    .groupby("season_start")
    ["game_date"]
    .min()
    .to_dict()
)

cutoff_rows = []

for target_year in sorted(
    snapshot["target_year"]
    .dropna()
    .astype(int)
    .unique()
):
    if (
        target_year
        in first_big5_game_by_year
    ):
        first_game = (
            first_big5_game_by_year[
                target_year
            ]
        )

        cutoff_date = (
            first_game
            - pd.Timedelta(days=1)
        )

        source = "duckdb_big5_first_game"

    else:
        first_game = pd.NaT

        cutoff_date = pd.Timestamp(
            year=int(target_year),
            month=FALLBACK_CUTOFF_MONTH,
            day=FALLBACK_CUTOFF_DAY,
        )

        source = "fallback_jul31"

    cutoff_rows.append({
        "target_year": int(target_year),
        "first_big5_game_date": first_game,
        "cutoff_date": cutoff_date,
        "cutoff_source": source,
    })


cutoff_table = pd.DataFrame(
    cutoff_rows
)

snapshot = snapshot.merge(
    cutoff_table,
    on="target_year",
    how="left",
    validate="many_to_one",
)

display(cutoff_table)

print("\nCutoff source counts")
display(
    cutoff_table[
        "cutoff_source"
    ]
    .value_counts()
    .rename("seasons")
    .to_frame()
)

,target_year,first_big5_game_date,cutoff_date,cutoff_source
0,2001,NaT,2001-07-31,fallback_jul31
1,2002,NaT,2002-07-31,fallback_jul31
2,2003,NaT,2003-07-31,fallback_jul31
3,2004,NaT,2004-07-31,fallback_jul31
4,2005,NaT,2005-07-31,fallback_jul31
5,2006,NaT,2006-07-31,fallback_jul31
6,2007,NaT,2007-07-31,fallback_jul31
7,2008,NaT,2008-07-31,fallback_jul31
8,2009,NaT,2009-07-31,fallback_jul31
9,2010,NaT,2010-07-31,fallback_jul31



Cutoff source counts


,seasons
cutoff_source,
duckdb_big5_first_game,13
fallback_jul31,11


# Part D. Summer Transfer Event 정리

## 17. eordo summer in/out → canonical transfer event

v2에서도 eordo 데이터는 다음 정보 때문에 유지합니다.

- summer / winter
- in / out
- fee
- is_loan
- transfer 당시 market value
- Big 5 쪽 league / country

다만 **날짜와 이동 순서의 기준은 DuckDB transfer history를 함께 사용**합니다.

`Laliga` 같은 리그 이름은 이미 `La Liga`로 canonicalize된 상태입니다.

In [19]:
summer = (
    eordo[
        eordo["window"]
        .astype(str)
        .str.lower()
        .eq("summer")
    ]
    .copy()
)

summer["movement_lower"] = (
    summer["movement"]
    .astype(str)
    .str.lower()
)

is_in = (
    summer["movement_lower"]
    .eq("in")
)

summer["from_club"] = np.where(
    is_in,
    summer["dealing_club"],
    summer["club"],
)

summer["to_club"] = np.where(
    is_in,
    summer["club"],
    summer["dealing_club"],
)

summer["from_league"] = np.where(
    is_in,
    np.nan,
    summer["league"],
)

summer["to_league"] = np.where(
    is_in,
    summer["league"],
    np.nan,
)

summer["from_country"] = np.where(
    is_in,
    summer["dealing_country"],
    summer["league"].map(
        LEAGUE_COUNTRY
    ),
)

summer["to_country"] = np.where(
    is_in,
    summer["league"].map(
        LEAGUE_COUNTRY
    ),
    summer["dealing_country"],
)

summer["from_league"] = (
    pd.Series(
        summer["from_league"],
        index=summer.index,
    )
    .map(canonicalize_league)
)

summer["to_league"] = (
    pd.Series(
        summer["to_league"],
        index=summer.index,
    )
    .map(canonicalize_league)
)

summer["norm_from_club"] = (
    summer["from_club"]
    .map(normalize_club)
)

summer["norm_to_club"] = (
    summer["to_club"]
    .map(normalize_club)
)

summer["market_value"] = pd.to_numeric(
    summer["market_value"],
    errors="coerce",
)

summer["fee"] = pd.to_numeric(
    summer["fee"],
    errors="coerce",
)

summer["is_loan"] = (
    pd.to_numeric(
        summer["is_loan"],
        errors="coerce",
    )
    .fillna(0)
    .astype(int)
)


def first_nonnull(series):
    series = series.dropna()

    if series.empty:
        return np.nan

    return series.iloc[0]


transfer_events = (
    summer
    .groupby(
        [
            "player_id",
            "season",
            "norm_from_club",
            "norm_to_club",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        player_name=(
            "player_name",
            first_nonnull,
        ),
        from_club=(
            "from_club",
            first_nonnull,
        ),
        to_club=(
            "to_club",
            first_nonnull,
        ),
        from_league=(
            "from_league",
            first_nonnull,
        ),
        to_league=(
            "to_league",
            first_nonnull,
        ),
        from_country=(
            "from_country",
            first_nonnull,
        ),
        to_country=(
            "to_country",
            first_nonnull,
        ),
        transfer_market_value_eur=(
            "market_value",
            "max",
        ),
        transfer_fee_eur=(
            "fee",
            "max",
        ),
        is_loan=(
            "is_loan",
            "max",
        ),
        has_big5_in_record=(
            "movement_lower",
            lambda s: int(
                (s == "in").any()
            ),
        ),
        has_big5_out_record=(
            "movement_lower",
            lambda s: int(
                (s == "out").any()
            ),
        ),
        source_rows=(
            "movement_lower",
            "size",
        ),
    )
)

transfer_events[
    "player_id"
] = (
    transfer_events[
        "player_id"
    ]
    .astype(int)
)

transfer_events[
    "season"
] = (
    transfer_events[
        "season"
    ]
    .astype(int)
)

transfer_events[
    "event_id"
] = np.arange(
    len(transfer_events),
    dtype=int,
)

print(
    "Canonical summer events:",
    len(transfer_events),
)

print(
    "Events represented by both in/out rows:",
    int(
        (
            transfer_events[
                "source_rows"
            ] > 1
        ).sum()
    ),
)

display(
    transfer_events.head()
)

Canonical summer events: 50672
Events represented by both in/out rows: 10071


,player_id,season,norm_from_club,norm_to_club,player_name,from_club,to_club,from_league,to_league,from_country,to_country,transfer_market_value_eur,transfer_fee_eur,is_loan,has_big5_in_record,has_big5_out_record,source_rows,event_id
0,1,2003,1 fc kaiserslautern,vfb lubeck,Silvio Adzic,1.FC Kaiserslautern,VfB Lübeck,Bundesliga,NaN,Germany,Germany,NaN,0.0,0,0,1,1,0
1,2,2003,1 fc kaiserslautern,al rayyan sc,Mario Basler,1.FC Kaiserslautern,Al-Rayyan SC,Bundesliga,NaN,Germany,Qatar,NaN,0.0,0,0,1,1,1
2,4,2004,bolton wanderers,without club,Youri Djorkaeff,Bolton Wanderers,Without Club,Premier League,NaN,England,NaN,NaN,NaN,0,0,1,1,2
3,4,2004,without club,blackburn rovers,Youri Djorkaeff,Without Club,Blackburn Rovers,NaN,Premier League,NaN,England,NaN,NaN,0,1,0,1,3
4,5,2000,ac sparta prague,1 fc kaiserslautern,Petr Gabriel,AC Sparta Prague,1.FC Kaiserslautern,NaN,Bundesliga,Czech Republic,Germany,NaN,1500000.0,0,1,0,1,4


## 18. v2 — eordo event와 DuckDB transfer date를 **1:1 assignment**로 연결

v1은 각 eordo event가 독립적으로 DuckDB 후보 하나를 고르는 방식이었습니다.

한 선수에게 같은 여름 여러 이벤트가 있으면:

```text
이벤트 A ─┐
          ├→ 같은 DB 후보를 서로 선택
이벤트 B ─┘
```

처럼 모호해질 수 있었습니다.

v2에서는 같은 `(player_id, target_year)` 안에서 모든 pair score를 계산한 뒤
**각 eordo event와 각 DuckDB transfer row가 최대 한 번씩만 사용되는 1:1 greedy assignment**를 합니다.

Club 비교도 `FC / CF / Olympique / Borussia` 같은 표기에 더 강한
`club_similarity()`를 사용합니다.

In [20]:
db_transfers = pd.DataFrame({
    "player_id": pd.to_numeric(
        db_transfers_raw[
            TRANSFER_PLAYER_ID_COL
        ],
        errors="coerce",
    ),
    "transfer_date": pd.to_datetime(
        db_transfers_raw[
            TRANSFER_DATE_COL
        ],
        errors="coerce",
    ),
    "db_from_club": (
        db_transfers_raw[
            TRANSFER_FROM_COL
        ]
        .astype(str)
    ),
    "db_to_club": (
        db_transfers_raw[
            TRANSFER_TO_COL
        ]
        .astype(str)
    ),
})

if TRANSFER_FEE_COL is not None:
    db_transfers[
        "db_transfer_fee_eur"
    ] = pd.to_numeric(
        db_transfers_raw[
            TRANSFER_FEE_COL
        ],
        errors="coerce",
    )
else:
    db_transfers[
        "db_transfer_fee_eur"
    ] = np.nan

if TRANSFER_MV_COL is not None:
    db_transfers[
        "db_market_value_eur"
    ] = pd.to_numeric(
        db_transfers_raw[
            TRANSFER_MV_COL
        ],
        errors="coerce",
    )
else:
    db_transfers[
        "db_market_value_eur"
    ] = np.nan


db_transfers = (
    db_transfers
    .dropna(
        subset=[
            "player_id",
            "transfer_date",
        ]
    )
    .copy()
)

db_transfers[
    "player_id"
] = (
    db_transfers[
        "player_id"
    ]
    .astype(int)
)

db_transfers[
    "transfer_year"
] = (
    db_transfers[
        "transfer_date"
    ]
    .dt.year
)

db_transfers[
    "norm_db_from"
] = (
    db_transfers[
        "db_from_club"
    ]
    .map(normalize_club)
)

db_transfers[
    "norm_db_to"
] = (
    db_transfers[
        "db_to_club"
    ]
    .map(normalize_club)
)

db_transfers[
    "db_row_id"
] = np.arange(
    len(db_transfers),
    dtype=int,
)

db_transfer_groups = {
    key: group.reset_index(
        drop=True
    )
    for key, group
    in db_transfers.groupby(
        [
            "player_id",
            "transfer_year",
        ]
    )
}


def transfer_pair_score(
    e_from,
    e_to,
    d_from,
    d_to,
):
    from_score = club_similarity(
        e_from,
        d_from,
    )

    to_score = club_similarity(
        e_to,
        d_to,
    )

    # 두 방향 모두 중요하므로 평균 사용
    return (
        from_score + to_score
    ) / 2.0


date_match_rows = []

# 같은 player-year 안에서 event들을 동시에 배정
for (
    player_id,
    season,
), ev_group in transfer_events.groupby(
    ["player_id", "season"]
):
    ev_group = ev_group.copy()

    db_group = db_transfer_groups.get(
        (
            int(player_id),
            int(season),
        )
    )

    # 기본 no-candidate 레코드
    if (
        db_group is None
        or db_group.empty
    ):
        for ev in ev_group.itertuples():
            date_match_rows.append({
                "event_id": ev.event_id,
                "player_id": ev.player_id,
                "season": ev.season,
                "from_club": ev.from_club,
                "to_club": ev.to_club,
                "transfer_date": pd.NaT,
                "db_match_score": np.nan,
                "db_date_match_method": "no_db_candidate",
                "db_transfer_fee_eur": np.nan,
                "db_market_value_eur": np.nan,
            })
        continue

    pair_candidates = []

    for ev in ev_group.itertuples():
        for db_row in db_group.itertuples():
            from_score = club_similarity(
                ev.from_club,
                db_row.db_from_club,
            )

            to_score = club_similarity(
                ev.to_club,
                db_row.db_to_club,
            )

            pair_score = (
                from_score + to_score
            ) / 2.0

            pair_candidates.append({
                "event_id": int(ev.event_id),
                "db_row_id": int(db_row.db_row_id),
                "pair_score": float(pair_score),
                "from_score": float(from_score),
                "to_score": float(to_score),
            })

    pair_candidates = sorted(
        pair_candidates,
        key=lambda x: (
            x["pair_score"],
            min(
                x["from_score"],
                x["to_score"],
            ),
        ),
        reverse=True,
    )

    assigned_events = set()
    assigned_db_rows = set()
    assignments = {}

    for pair in pair_candidates:
        event_id = pair["event_id"]
        db_row_id = pair["db_row_id"]

        if event_id in assigned_events:
            continue

        if db_row_id in assigned_db_rows:
            continue

        strong_direction = (
            pair["from_score"] >= 0.82
            or pair["to_score"] >= 0.82
        )

        acceptable = (
            pair["pair_score"]
            >= TRANSFER_PAIR_STRONG_SCORE
            or (
                pair["pair_score"]
                >= TRANSFER_PAIR_MIN_SCORE
                and strong_direction
            )
        )

        if not acceptable:
            continue

        assignments[event_id] = pair

        assigned_events.add(
            event_id
        )

        assigned_db_rows.add(
            db_row_id
        )

    for ev in ev_group.itertuples():
        base = {
            "event_id": ev.event_id,
            "player_id": ev.player_id,
            "season": ev.season,
            "from_club": ev.from_club,
            "to_club": ev.to_club,
        }

        pair = assignments.get(
            int(ev.event_id)
        )

        if pair is None:
            best_scores = [
                p
                for p in pair_candidates
                if p["event_id"]
                == int(ev.event_id)
            ]

            best_score = (
                best_scores[0]["pair_score"]
                if best_scores
                else np.nan
            )

            date_match_rows.append({
                **base,
                "transfer_date": pd.NaT,
                "db_match_score": best_score,
                "db_date_match_method": "unresolved_after_1to1_assignment",
                "db_transfer_fee_eur": np.nan,
                "db_market_value_eur": np.nan,
            })

            continue

        db_match = db_group.loc[
            db_group[
                "db_row_id"
            ].eq(
                pair["db_row_id"]
            )
        ].iloc[0]

        exact_pair = (
            normalize_club(
                ev.from_club
            )
            == normalize_club(
                db_match[
                    "db_from_club"
                ]
            )
            and normalize_club(
                ev.to_club
            )
            == normalize_club(
                db_match[
                    "db_to_club"
                ]
            )
        )

        if exact_pair:
            method = (
                "exact_player_year_club_pair"
            )
        elif (
            pair["pair_score"]
            >= TRANSFER_PAIR_STRONG_SCORE
        ):
            method = (
                "strong_1to1_club_pair"
            )
        else:
            method = (
                "directional_1to1_club_pair"
            )

        date_match_rows.append({
            **base,
            "transfer_date": db_match[
                "transfer_date"
            ],
            "db_match_score": pair[
                "pair_score"
            ],
            "db_date_match_method": method,
            "db_transfer_fee_eur": db_match[
                "db_transfer_fee_eur"
            ],
            "db_market_value_eur": db_match[
                "db_market_value_eur"
            ],
        })


transfer_date_matches = pd.DataFrame(
    date_match_rows
)

transfer_events = (
    transfer_events
    .merge(
        transfer_date_matches[
            [
                "event_id",
                "transfer_date",
                "db_match_score",
                "db_date_match_method",
                "db_transfer_fee_eur",
                "db_market_value_eur",
            ]
        ],
        on="event_id",
        how="left",
        validate="one_to_one",
    )
)

# eordo 값을 우선 사용하고 없을 때 DB 값 fallback
transfer_events[
    "transfer_fee_eur"
] = (
    transfer_events[
        "transfer_fee_eur"
    ]
    .combine_first(
        transfer_events[
            "db_transfer_fee_eur"
        ]
    )
)

transfer_events[
    "transfer_market_value_eur"
] = (
    transfer_events[
        "transfer_market_value_eur"
    ]
    .combine_first(
        transfer_events[
            "db_market_value_eur"
        ]
    )
)

transfer_date_match_rate = (
    transfer_events[
        "transfer_date"
    ]
    .notna()
    .mean()
)

transfer_date_coverage_by_year = (
    transfer_events
    .assign(
        date_known=lambda d:
            d["transfer_date"].notna()
    )
    .groupby("season")
    .agg(
        events=("event_id", "size"),
        dated_events=(
            "date_known",
            "sum",
        ),
        date_coverage=(
            "date_known",
            "mean",
        ),
    )
    .reset_index()
)

print(
    "Summer event exact-date coverage:",
    f"{transfer_date_match_rate:.2%}"
)

display(
    transfer_events[
        "db_date_match_method"
    ]
    .value_counts(
        dropna=False
    )
    .rename("n")
    .to_frame()
)

display(
    transfer_date_coverage_by_year.tail(15)
)

Summer event exact-date coverage: 27.86%


,n
db_date_match_method,
no_db_candidate,36205
strong_1to1_club_pair,11938
exact_player_year_club_pair,1483
directional_1to1_club_pair,694
unresolved_after_1to1_assignment,352


,season,events,dated_events,date_coverage
11,2011,2089,194,0.092867
12,2012,2092,231,0.110421
13,2013,2274,340,0.149516
14,2014,2405,472,0.196258
15,2015,2302,615,0.267159
16,2016,2227,686,0.308038
17,2017,2149,792,0.368544
18,2018,2133,919,0.430849
19,2019,2225,1069,0.480449
20,2020,1919,1022,0.532569


## 19. v2 — 한 선수의 summer transfer를 **시간순 team-state transition**으로 추적

v1에서는 player-season마다 transfer event 하나만 골랐습니다.

하지만 실제 이적은 한 여름에 여러 이벤트가 연속될 수 있습니다.

예: Guirassy 2023

```text
관측 현재 팀: Stuttgart

06-30 Stuttgart → Rennes   (임대 복귀)
07-01 Rennes → Stuttgart   (완전 이적)

최종 preseason 팀: Stuttgart
```

v1 방식은 이벤트 하나만 잡으면서 Rennes로 잘못 이동시킬 수 있었습니다.

v2는:

1. 현재 팀에서 시작
2. cutoff 이전 dated event를 날짜순 정렬
3. 현재 team state와 `from_club`이 맞으면 실제 이동 적용
4. `to_club`이 현재 team state와 맞으면 계약 전환/입단 처리로 보고 팀은 유지
5. 다음 event에서 갱신된 team state를 다시 사용
6. cutoff 시점의 최종 team state를 destination으로 사용

합니다.

In [21]:
events_by_player_year = {
    key: group.sort_values(
        [
            "transfer_date",
            "event_id",
        ],
        na_position="last",
    ).reset_index(
        drop=True
    )
    for key, group
    in transfer_events.groupby(
        [
            "player_id",
            "season",
        ]
    )
}


def trace_preseason_transfer_timeline(
    current_team,
    current_league,
    player_id,
    target_year,
    cutoff_date,
):
    """
    cutoff 이전 dated event만 실제 feature에 반영.

    unknown-date / post-cutoff event는
    audit count로만 반환하고 model feature에는 쓰지 않음.
    """
    base_country = (
        LEAGUE_COUNTRY.get(
            current_league,
            np.nan,
        )
    )

    result = {
        "timeline_status": None,
        "confirmed_preseason_event_count": 0,
        "preseason_team_transition_count": 0,
        "selected_event_id": np.nan,
        "destination_team": current_team,
        "destination_league": current_league,
        "destination_country": base_country,
        "unknown_date_context_event_count_audit": 0,
        "post_cutoff_context_event_count_audit": 0,
        "context_mismatch_pre_event_count_audit": 0,
        "timeline_trace": "",
    }

    if pd.isna(player_id):
        result[
            "timeline_status"
        ] = "no_player_id"

        return result

    key = (
        int(player_id),
        int(target_year),
    )

    candidates = (
        events_by_player_year.get(
            key
        )
    )

    if (
        candidates is None
        or candidates.empty
    ):
        result[
            "timeline_status"
        ] = "no_summer_event"

        return result

    cutoff_date = pd.Timestamp(
        cutoff_date
    )

    known = candidates.loc[
        candidates[
            "transfer_date"
        ].notna()
    ].copy()

    unknown = candidates.loc[
        candidates[
            "transfer_date"
        ].isna()
    ].copy()

    pre = known.loc[
        known[
            "transfer_date"
        ]
        <= cutoff_date
    ].sort_values(
        [
            "transfer_date",
            "event_id",
        ]
    )

    post = known.loc[
        known[
            "transfer_date"
        ]
        > cutoff_date
    ].sort_values(
        [
            "transfer_date",
            "event_id",
        ]
    )

    # unknown/post event는 audit만.
    # "미래에 event가 존재한다"는 사실 자체를
    # model feature로 넣으면 leakage가 될 수 있기 때문.
    for _, ev in unknown.iterrows():
        if max(
            club_similarity(
                current_team,
                ev["from_club"],
            ),
            club_similarity(
                current_team,
                ev["to_club"],
            ),
        ) >= TEAM_CONTEXT_MIN_SCORE:
            result[
                "unknown_date_context_event_count_audit"
            ] += 1

    for _, ev in post.iterrows():
        if max(
            club_similarity(
                current_team,
                ev["from_club"],
            ),
            club_similarity(
                current_team,
                ev["to_club"],
            ),
        ) >= TEAM_CONTEXT_MIN_SCORE:
            result[
                "post_cutoff_context_event_count_audit"
            ] += 1

    team_state = current_team
    league_state = current_league
    country_state = base_country

    trace_parts = []
    relevant_events = []

    for _, ev in pre.iterrows():
        from_score = club_similarity(
            team_state,
            ev["from_club"],
        )

        to_score = club_similarity(
            team_state,
            ev["to_club"],
        )

        # A) 현재 team state에서 나가는 event
        if (
            from_score
            >= TEAM_CONTEXT_MIN_SCORE
        ):
            old_state = team_state

            team_state = ev[
                "to_club"
            ]

            if pd.notna(
                ev["to_league"]
            ):
                league_state = (
                    canonicalize_league(
                        ev["to_league"]
                    )
                )
            else:
                # Big5 outbound인데 destination league가
                # 없는 경우는 Big5 밖으로 이동한 것으로 처리.
                league_state = np.nan

            if pd.notna(
                ev["to_country"]
            ):
                country_state = ev[
                    "to_country"
                ]
            elif (
                pd.notna(league_state)
                and league_state
                in LEAGUE_COUNTRY
            ):
                country_state = (
                    LEAGUE_COUNTRY[
                        league_state
                    ]
                )
            else:
                country_state = np.nan

            result[
                "confirmed_preseason_event_count"
            ] += 1

            # 최종적으로 같은 이름 변형이 아닌
            # 실질 팀 이동일 때 transition count 증가
            if (
                club_similarity(
                    old_state,
                    team_state,
                )
                < SAME_CLUB_SCORE
            ):
                result[
                    "preseason_team_transition_count"
                ] += 1

            relevant_events.append(
                ev
            )

            trace_parts.append(
                f"{ev['transfer_date'].date()}: "
                f"{old_state} -> {team_state}"
            )

            continue

        # B) 현재 team state로 들어오는 event
        #    예: 임대 복귀 직후 완전 이적.
        #    실제 경기 팀은 이미 현재 팀이므로 state 유지.
        if (
            to_score
            >= TEAM_CONTEXT_MIN_SCORE
        ):
            result[
                "confirmed_preseason_event_count"
            ] += 1

            relevant_events.append(
                ev
            )

            # league/country가 알려져 있으면
            # 현재 팀과 일관된 정보로 갱신 가능
            if pd.notna(
                ev["to_league"]
            ):
                league_state = (
                    canonicalize_league(
                        ev["to_league"]
                    )
                )

            if pd.notna(
                ev["to_country"]
            ):
                country_state = ev[
                    "to_country"
                ]

            trace_parts.append(
                f"{ev['transfer_date'].date()}: "
                f"arrival/contract -> {team_state}"
            )

            continue

        result[
            "context_mismatch_pre_event_count_audit"
        ] += 1

    if relevant_events:
        last_event = (
            relevant_events[-1]
        )

        result[
            "selected_event_id"
        ] = int(
            last_event[
                "event_id"
            ]
        )

        result[
            "timeline_status"
        ] = (
            "confirmed_pre_cutoff_event"
        )

    else:
        result[
            "timeline_status"
        ] = (
            "no_confirmed_pre_cutoff_event"
        )

    result[
        "destination_team"
    ] = team_state

    result[
        "destination_league"
    ] = league_state

    result[
        "destination_country"
    ] = country_state

    result[
        "timeline_trace"
    ] = " | ".join(
        trace_parts
    )

    return result


timeline_rows = []

for row in snapshot[
    [
        "row_id",
        "team",
        "league",
        "tm_player_id",
        "target_year",
        "cutoff_date",
    ]
].itertuples(
    index=False
):
    result = (
        trace_preseason_transfer_timeline(
            current_team=row.team,
            current_league=row.league,
            player_id=row.tm_player_id,
            target_year=row.target_year,
            cutoff_date=row.cutoff_date,
        )
    )

    timeline_rows.append({
        "row_id": row.row_id,
        **result,
    })


transfer_timeline = pd.DataFrame(
    timeline_rows
)

snapshot = snapshot.merge(
    transfer_timeline,
    on="row_id",
    how="left",
    validate="one_to_one",
)

# 최종 economics는 cutoff 이전 관련 event 중
# 마지막 event의 값을 사용
selected_event_features = (
    transfer_events[
        [
            "event_id",
            "transfer_date",
            "from_club",
            "to_club",
            "from_league",
            "to_league",
            "from_country",
            "to_country",
            "transfer_fee_eur",
            "transfer_market_value_eur",
            "is_loan",
            "db_date_match_method",
            "db_match_score",
        ]
    ]
    .add_prefix("tr_")
)

snapshot = snapshot.merge(
    selected_event_features,
    left_on="selected_event_id",
    right_on="tr_event_id",
    how="left",
    validate="many_to_one",
)


# --------------------------------------------------------------
# Audit 파일: model feature가 아님
# --------------------------------------------------------------

unknown_date_audit_rows = []
post_cutoff_audit_rows = []

for row in snapshot[
    [
        "row_id",
        "player",
        "season",
        "target_season",
        "team",
        "tm_player_id",
        "target_year",
        "cutoff_date",
    ]
].itertuples(
    index=False
):
    if pd.isna(
        row.tm_player_id
    ):
        continue

    candidates = (
        events_by_player_year.get(
            (
                int(row.tm_player_id),
                int(row.target_year),
            )
        )
    )

    if (
        candidates is None
        or candidates.empty
    ):
        continue

    for ev in candidates.itertuples():
        context_score = max(
            club_similarity(
                row.team,
                ev.from_club,
            ),
            club_similarity(
                row.team,
                ev.to_club,
            ),
        )

        if (
            context_score
            < TEAM_CONTEXT_MIN_SCORE
        ):
            continue

        if pd.isna(
            ev.transfer_date
        ):
            unknown_date_audit_rows.append({
                "row_id": row.row_id,
                "player": row.player,
                "season": row.season,
                "target_season": row.target_season,
                "team": row.team,
                "cutoff_date": row.cutoff_date,
                "from_club": ev.from_club,
                "to_club": ev.to_club,
                "context_score": context_score,
                "db_date_match_method": ev.db_date_match_method,
            })

        elif (
            pd.Timestamp(
                ev.transfer_date
            )
            > pd.Timestamp(
                row.cutoff_date
            )
        ):
            post_cutoff_audit_rows.append({
                "row_id": row.row_id,
                "player": row.player,
                "season": row.season,
                "target_season": row.target_season,
                "team": row.team,
                "cutoff_date": row.cutoff_date,
                "transfer_date": ev.transfer_date,
                "from_club": ev.from_club,
                "to_club": ev.to_club,
                "context_score": context_score,
            })


unknown_date_transfer_audit = (
    pd.DataFrame(
        unknown_date_audit_rows
    )
)

post_cutoff_transfer_audit = (
    pd.DataFrame(
        post_cutoff_audit_rows
    )
)

print(
    "Rows with confirmed pre-cutoff event:",
    int(
        (
            snapshot[
                "confirmed_preseason_event_count"
            ] > 0
        ).sum()
    ),
)

print(
    "Rows whose final team changed:",
    int(
        (
            snapshot[
                "preseason_team_transition_count"
            ] > 0
        ).sum()
    ),
)

print(
    "Unknown-date context events (audit only):",
    len(
        unknown_date_transfer_audit
    ),
)

print(
    "Post-cutoff context events (audit only):",
    len(
        post_cutoff_transfer_audit
    ),
)

display(
    snapshot.loc[
        snapshot[
            "confirmed_preseason_event_count"
        ] > 0,
        [
            "player",
            "season",
            "team",
            "cutoff_date",
            "confirmed_preseason_event_count",
            "preseason_team_transition_count",
            "destination_team",
            "timeline_trace",
        ],
    ].head(20)
)

Rows with confirmed pre-cutoff event: 1287
Rows whose final team changed: 1271
Unknown-date context events (audit only): 4090
Post-cutoff context events (audit only): 552


,player,season,team,cutoff_date,confirmed_preseason_event_count,preseason_team_transition_count,destination_team,timeline_trace
3105,James Milner,2003-2004,Leeds United,2004-07-31,1,1,Newcastle United,2004-07-02: Leeds United -> Newcastle United
5152,Lukas Podolski,2005-2006,Köln,2006-07-31,1,1,Bayern Munich,2006-07-10: Köln -> Bayern Munich
5407,Santi Cazorla,2005-2006,Villarreal,2006-07-31,1,1,Recreativo Huelva,2006-07-07: Villarreal -> Recreativo Huelva
5579,André-Pierre Gignac,2006-2007,Lorient,2007-07-31,1,1,FC Toulouse,2007-07-01: Lorient -> FC Toulouse
6326,Raúl García,2006-2007,Osasuna,2007-07-31,1,1,Atlético de Madrid,2007-07-01: Osasuna -> Atlético de Madrid
6375,Santi Cazorla,2006-2007,Recreativo,2007-07-31,1,1,Villarreal CF,2007-07-01: Recreativo -> Villarreal CF
6430,Steven Davis,2006-2007,Aston Villa,2007-07-31,1,1,Fulham FC,2007-07-01: Aston Villa -> Fulham FC
7073,Kévin Gameiro,2007-2008,Strasbourg,2008-07-31,1,1,FC Lorient,2008-07-01: Strasbourg -> FC Lorient
7231,Miralem Pjanić,2007-2008,Metz,2008-07-31,1,1,Olympique Lyon,2008-07-01: Metz -> Olympique Lyon
7554,Álvaro Negredo,2008-2009,Almería,2009-07-31,1,1,Real Madrid,2009-07-01: Almería -> Real Madrid


## 20. v2 — Leakage-safe Transfer Feature 생성

### 의미를 명확하게 바꿉니다.

`transfer_event_preseason=1`은 이제:

> **우리의 dated historical source에서 cutoff 이전이라고 확인된 관련 transfer event가 존재한다**

를 의미합니다.

날짜를 확인하지 못한 summer event는 0으로 '이적 없음'이라고 확정하는 것이 아니라
**모델 feature에는 사용하지 않고 audit에만 남깁니다.**

또한:

- transfer event가 있었지만 최종 팀이 그대로일 수 있음
- 여러 번 이동 후 원래 팀으로 돌아올 수도 있음

따라서 event와 최종 team change를 분리합니다.

In [22]:
snapshot[
    "transfer_event_preseason"
] = (
    snapshot[
        "confirmed_preseason_event_count"
    ]
    .fillna(0)
    .gt(0)
    .astype(int)
)

snapshot[
    "norm_destination_team"
] = (
    snapshot[
        "destination_team"
    ]
    .map(normalize_club)
)

# 최종 cutoff 시점 팀이 원래 팀과 다른가?
snapshot[
    "changed_team_preseason"
] = (
    snapshot.apply(
        lambda r: int(
            club_similarity(
                r["team"],
                r[
                    "destination_team"
                ],
            )
            < SAME_CLUB_SCORE
        ),
        axis=1,
    )
)

# league 표기 다시 canonicalize
snapshot[
    "destination_league"
] = (
    snapshot[
        "destination_league"
    ]
    .map(canonicalize_league)
)

# 이적이 없거나 최종 팀이 그대로면
# 현재 Big5 league를 유지
same_final_team = (
    snapshot[
        "changed_team_preseason"
    ].eq(0)
)

snapshot.loc[
    same_final_team,
    "destination_league",
] = snapshot.loc[
    same_final_team,
    "league",
]

snapshot.loc[
    same_final_team,
    "destination_country",
] = (
    snapshot.loc[
        same_final_team,
        "league",
    ]
    .map(
        LEAGUE_COUNTRY
    )
)

snapshot[
    "destination_in_big5"
] = (
    snapshot[
        "destination_league"
    ]
    .isin(
        BIG5_LEAGUES
    )
    .astype(int)
)

# 같은 리그 이적
snapshot[
    "same_league_transfer"
] = 0

changed_mask = (
    snapshot[
        "changed_team_preseason"
    ].eq(1)
)

snapshot.loc[
    changed_mask,
    "same_league_transfer",
] = (
    snapshot.loc[
        changed_mask,
        "destination_league",
    ]
    .eq(
        snapshot.loc[
            changed_mask,
            "league",
        ]
    )
    .astype(int)
)

snapshot[
    "league_changed"
] = (
    changed_mask
    & snapshot[
        "same_league_transfer"
    ].eq(0)
).astype(int)

current_country = (
    snapshot[
        "league"
    ]
    .map(
        LEAGUE_COUNTRY
    )
)

snapshot[
    "country_changed"
] = 0

country_known = (
    changed_mask
    & snapshot[
        "destination_country"
    ].notna()
)

snapshot.loc[
    country_known,
    "country_changed",
] = (
    snapshot.loc[
        country_known,
        "destination_country",
    ]
    .astype(str)
    .str.lower()
    .ne(
        current_country.loc[
            country_known
        ]
        .astype(str)
        .str.lower()
    )
    .astype(int)
)

snapshot[
    "is_loan_preseason"
] = 0

selected_event_mask = (
    snapshot[
        "selected_event_id"
    ].notna()
)

snapshot.loc[
    selected_event_mask,
    "is_loan_preseason",
] = (
    snapshot.loc[
        selected_event_mask,
        "tr_is_loan",
    ]
    .fillna(0)
    .astype(int)
)

snapshot[
    "transfer_date_preseason"
] = pd.NaT

snapshot.loc[
    selected_event_mask,
    "transfer_date_preseason",
] = (
    snapshot.loc[
        selected_event_mask,
        "tr_transfer_date",
    ]
)

snapshot[
    "days_since_transfer"
] = (
    snapshot[
        "cutoff_date"
    ]
    - snapshot[
        "transfer_date_preseason"
    ]
).dt.days

snapshot[
    "transfer_fee_eur"
] = np.nan

snapshot.loc[
    selected_event_mask,
    "transfer_fee_eur",
] = snapshot.loc[
    selected_event_mask,
    "tr_transfer_fee_eur",
]

snapshot[
    "transfer_market_value_eur"
] = np.nan

snapshot.loc[
    selected_event_mask,
    "transfer_market_value_eur",
] = snapshot.loc[
    selected_event_mask,
    "tr_transfer_market_value_eur",
]

snapshot[
    "transfer_fee_known"
] = (
    snapshot[
        "transfer_fee_eur"
    ].notna()
    & selected_event_mask
).astype(int)

snapshot[
    "transfer_fee_positive"
] = (
    snapshot[
        "transfer_fee_eur"
    ]
    .fillna(0)
    .gt(0)
    .astype(int)
)

snapshot[
    "log_transfer_fee"
] = np.log1p(
    snapshot[
        "transfer_fee_eur"
    ]
    .clip(lower=0)
)

snapshot[
    "fee_to_transfer_market_value_ratio"
] = np.where(
    snapshot[
        "transfer_market_value_eur"
    ] > 0,
    (
        snapshot[
            "transfer_fee_eur"
        ]
        / snapshot[
            "transfer_market_value_eur"
        ]
    ),
    np.nan,
)

display(
    snapshot.loc[
        snapshot[
            "transfer_event_preseason"
        ].eq(1),
        [
            "player",
            "season",
            "team",
            "destination_team",
            "transfer_date_preseason",
            "cutoff_date",
            "confirmed_preseason_event_count",
            "preseason_team_transition_count",
            "changed_team_preseason",
            "destination_league",
            "destination_in_big5",
            "is_loan_preseason",
            "transfer_fee_eur",
        ],
    ].head(25)
)

,player,season,team,destination_team,transfer_date_preseason,cutoff_date,confirmed_preseason_event_count,preseason_team_transition_count,changed_team_preseason,destination_league,destination_in_big5,is_loan_preseason,transfer_fee_eur
3105,James Milner,2003-2004,Leeds United,Newcastle United,2004-07-02,2004-07-31,1,1,1,Premier League,1,0,7400000.0
5152,Lukas Podolski,2005-2006,Köln,Bayern Munich,2006-07-10,2006-07-31,1,1,1,Bundesliga,1,0,10000000.0
5407,Santi Cazorla,2005-2006,Villarreal,Recreativo Huelva,2006-07-07,2006-07-31,1,1,1,La Liga,1,0,400000.0
5579,André-Pierre Gignac,2006-2007,Lorient,FC Toulouse,2007-07-01,2007-07-31,1,1,1,Ligue 1,1,0,4500000.0
6326,Raúl García,2006-2007,Osasuna,Atlético de Madrid,2007-07-01,2007-07-31,1,1,1,La Liga,1,0,13000000.0
6375,Santi Cazorla,2006-2007,Recreativo,Villarreal CF,2007-07-01,2007-07-31,1,1,1,La Liga,1,0,1200000.0
6430,Steven Davis,2006-2007,Aston Villa,Fulham FC,2007-07-01,2007-07-31,1,1,1,Premier League,1,0,5900000.0
7073,Kévin Gameiro,2007-2008,Strasbourg,FC Lorient,2008-07-01,2008-07-31,1,1,1,Ligue 1,1,0,3000000.0
7231,Miralem Pjanić,2007-2008,Metz,Olympique Lyon,2008-07-01,2008-07-31,1,1,1,Ligue 1,1,0,7500000.0
7554,Álvaro Negredo,2008-2009,Almería,Real Madrid,2009-07-01,2009-07-31,1,1,1,La Liga,1,0,5000000.0


# Part E. Historical Market Value — Point-in-time Join

## 21. player_valuations 정리

시장가치에서 가장 중요한 원칙:

```text
valuation_date <= cutoff_date
```

현재 최신 시장가치를 과거 모든 시즌에 붙이면 심각한 미래 누수입니다.

아래에서는 각 선수-시즌 row마다:

- cutoff 직전 최신 시장가치
- cutoff - 6개월 시점에서 알 수 있었던 최신 시장가치
- cutoff - 12개월 시점에서 알 수 있었던 최신 시장가치
- cutoff까지의 역사적 peak

를 계산합니다.

In [23]:
valuations = pd.DataFrame({
    "tm_player_id": pd.to_numeric(
        valuations_raw[
            VALUATION_PLAYER_ID_COL
        ],
        errors="coerce",
    ),
    "valuation_date": pd.to_datetime(
        valuations_raw[
            VALUATION_DATE_COL
        ],
        errors="coerce",
    ),
    "market_value_eur": pd.to_numeric(
        valuations_raw[
            VALUATION_VALUE_COL
        ],
        errors="coerce",
    ),
})

valuations = (
    valuations
    .dropna(
        subset=[
            "tm_player_id",
            "valuation_date",
            "market_value_eur",
        ]
    )
    .copy()
)

valuations[
    "tm_player_id"
] = (
    valuations[
        "tm_player_id"
    ]
    .astype(int)
)

valuations = (
    valuations
    .sort_values(
        [
            "tm_player_id",
            "valuation_date",
        ]
    )
    .drop_duplicates(
        subset=[
            "tm_player_id",
            "valuation_date",
        ],
        keep="last",
    )
)

valuations[
    "market_value_peak_to_date_eur"
] = (
    valuations
    .groupby(
        "tm_player_id"
    )
    ["market_value_eur"]
    .cummax()
)

print(
    "Valuation rows:",
    len(valuations),
)

print(
    "Date range:",
    valuations["valuation_date"].min(),
    "~",
    valuations["valuation_date"].max(),
)

Valuation rows: 656301
Date range: 2000-01-20 00:00:00 ~ 2026-06-12 00:00:00


## 22. Generic point-in-time lookup 함수

In [24]:
def point_in_time_valuation_lookup(
    base_df,
    query_date_series,
    prefix,
):
    left = base_df[
        [
            "row_id",
            "tm_player_id",
        ]
    ].copy()

    left["query_date"] = pd.to_datetime(
        query_date_series,
        errors="coerce",
    )

    valid_left = (
        left[
            "tm_player_id"
        ].notna()
        & left[
            "query_date"
        ].notna()
    )

    query = (
        left.loc[
            valid_left
        ]
        .copy()
    )

    if query.empty:
        return pd.DataFrame({
            "row_id": base_df["row_id"],
            f"{prefix}_value_eur": np.nan,
            f"{prefix}_valuation_date": pd.NaT,
            f"{prefix}_peak_eur": np.nan,
        })

    query[
        "tm_player_id"
    ] = (
        query[
            "tm_player_id"
        ]
        .astype(int)
    )

    right = valuations[
        [
            "tm_player_id",
            "valuation_date",
            "market_value_eur",
            "market_value_peak_to_date_eur",
        ]
    ].copy()

    # merge_asof는 on key가 정렬되어 있어야 함
    query = query.sort_values(
        [
            "query_date",
            "tm_player_id",
        ]
    )

    right = right.sort_values(
        [
            "valuation_date",
            "tm_player_id",
        ]
    )

    merged = pd.merge_asof(
        query,
        right,
        left_on="query_date",
        right_on="valuation_date",
        by="tm_player_id",
        direction="backward",
        allow_exact_matches=True,
    )

    result = (
        merged[
            [
                "row_id",
                "market_value_eur",
                "valuation_date",
                "market_value_peak_to_date_eur",
            ]
        ]
        .rename(
            columns={
                "market_value_eur": f"{prefix}_value_eur",
                "valuation_date": f"{prefix}_valuation_date",
                "market_value_peak_to_date_eur": f"{prefix}_peak_eur",
            }
        )
    )

    all_rows = pd.DataFrame({
        "row_id": base_df[
            "row_id"
        ]
    })

    result = all_rows.merge(
        result,
        on="row_id",
        how="left",
        validate="one_to_one",
    )

    return result

## 23. cutoff / 6개월 전 / 12개월 전 시장가치 붙이기

In [25]:
mv_now = point_in_time_valuation_lookup(
    snapshot,
    snapshot["cutoff_date"],
    prefix="mv_preseason",
)

mv_6m = point_in_time_valuation_lookup(
    snapshot,
    (
        snapshot["cutoff_date"]
        - pd.Timedelta(days=183)
    ),
    prefix="mv_6m",
)

mv_12m = point_in_time_valuation_lookup(
    snapshot,
    (
        snapshot["cutoff_date"]
        - pd.Timedelta(days=365)
    ),
    prefix="mv_12m",
)

for table in [
    mv_now,
    mv_6m,
    mv_12m,
]:
    snapshot = snapshot.merge(
        table,
        on="row_id",
        how="left",
        validate="one_to_one",
    )

print(
    "Raw preseason valuation coverage:",
    f"{snapshot['mv_preseason_value_eur'].notna().mean():.2%}"
)

Raw preseason valuation coverage: 68.16%


## 24. Market Value Feature 생성

cutoff 직전 valuation이 없지만
cutoff 이전에 실제 이적이 있었고 그 event에 market value가 있다면
그 값을 fallback으로 사용할 수 있습니다.

다만 provenance를 남깁니다.

```text
market_value_source
= historical_valuation
  / transfer_event_fallback
  / missing
```

In [26]:
snapshot[
    "market_value_preseason_eur"
] = snapshot[
    "mv_preseason_value_eur"
]

snapshot[
    "market_value_source"
] = np.where(
    snapshot[
        "mv_preseason_value_eur"
    ].notna(),
    "historical_valuation",
    "missing",
)

fallback_mv_mask = (
    snapshot[
        "market_value_preseason_eur"
    ].isna()
    & snapshot[
        "transfer_event_preseason"
    ].eq(1)
    & snapshot[
        "transfer_market_value_eur"
    ].notna()
)

snapshot.loc[
    fallback_mv_mask,
    "market_value_preseason_eur",
] = snapshot.loc[
    fallback_mv_mask,
    "transfer_market_value_eur",
]

snapshot.loc[
    fallback_mv_mask,
    "market_value_source",
] = "transfer_event_fallback"

snapshot[
    "market_value_known"
] = (
    snapshot[
        "market_value_preseason_eur"
    ].notna()
).astype(int)

snapshot[
    "log_market_value"
] = np.log1p(
    snapshot[
        "market_value_preseason_eur"
    ].clip(lower=0)
)

snapshot[
    "market_value_6m_eur"
] = snapshot[
    "mv_6m_value_eur"
]

snapshot[
    "market_value_12m_eur"
] = snapshot[
    "mv_12m_value_eur"
]

snapshot[
    "market_value_change_6m_eur"
] = (
    snapshot[
        "market_value_preseason_eur"
    ]
    - snapshot[
        "market_value_6m_eur"
    ]
)

snapshot[
    "market_value_change_12m_eur"
] = (
    snapshot[
        "market_value_preseason_eur"
    ]
    - snapshot[
        "market_value_12m_eur"
    ]
)

snapshot[
    "market_value_growth_6m"
] = np.where(
    snapshot[
        "market_value_6m_eur"
    ] > 0,
    (
        snapshot[
            "market_value_preseason_eur"
        ]
        / snapshot[
            "market_value_6m_eur"
        ]
        - 1
    ),
    np.nan,
)

snapshot[
    "market_value_growth_12m"
] = np.where(
    snapshot[
        "market_value_12m_eur"
    ] > 0,
    (
        snapshot[
            "market_value_preseason_eur"
        ]
        / snapshot[
            "market_value_12m_eur"
        ]
        - 1
    ),
    np.nan,
)

snapshot[
    "market_value_peak_to_cutoff_eur"
] = snapshot[
    "mv_preseason_peak_eur"
]

snapshot[
    "market_value_vs_peak"
] = np.where(
    snapshot[
        "market_value_peak_to_cutoff_eur"
    ] > 0,
    (
        snapshot[
            "market_value_preseason_eur"
        ]
        / snapshot[
            "market_value_peak_to_cutoff_eur"
        ]
    ),
    np.nan,
)

snapshot[
    "valuation_age_days"
] = (
    snapshot[
        "cutoff_date"
    ]
    - snapshot[
        "mv_preseason_valuation_date"
    ]
).dt.days

# 같은 target season 내 상대적 market value
snapshot[
    "market_value_percentile"
] = (
    snapshot
    .groupby(
        "target_season"
    )
    ["market_value_preseason_eur"]
    .rank(
        pct=True,
        method="average",
    )
)

position_median = (
    snapshot
    .groupby(
        [
            "target_season",
            "position_group",
        ]
    )
    ["market_value_preseason_eur"]
    .transform("median")
)

snapshot[
    "market_value_vs_position_median"
] = np.where(
    position_median > 0,
    (
        snapshot[
            "market_value_preseason_eur"
        ]
        / position_median
    ),
    np.nan,
)

market_value_snapshots = (
    snapshot[
        [
            "row_id",
            "player",
            "season",
            "target_season",
            "tm_player_id",
            "cutoff_date",
            "market_value_preseason_eur",
            "market_value_source",
            "mv_preseason_valuation_date",
            "market_value_6m_eur",
            "mv_6m_valuation_date",
            "market_value_12m_eur",
            "mv_12m_valuation_date",
            "market_value_peak_to_cutoff_eur",
            "market_value_growth_6m",
            "market_value_growth_12m",
            "market_value_vs_peak",
        ]
    ]
    .copy()
)

display(
    snapshot[
        [
            "target_season",
            "market_value_known",
        ]
    ]
    .groupby("target_season")
    .agg(
        n=("market_value_known", "size"),
        coverage=("market_value_known", "mean"),
    )
    .tail(15)
)

,n,coverage
target_season,,
2010-2011,1019,0.722277
2011-2012,1009,0.802775
2012-2013,1042,0.888676
2013-2014,1063,0.948260
2014-2015,1065,0.953991
2015-2016,1093,0.949680
2016-2017,991,0.954591
2017-2018,993,0.952669
2018-2019,954,0.949686


# Part F. New Team Environment

## 25. 현재 팀 직전 시즌 성적 연결

현재 선수 row가 2023-24라면
2023-24 팀 최종 성적은 시즌 종료 후 이미 알고 있는 정보입니다.

기존 Exp9Cb에서도 사용했던:

- team_rank_pct
- team_points_per_game
- team_goal_diff_per_game

를 old team 기준으로 다시 붙입니다.

In [27]:
alias_map = dict(
    zip(
        team_alias[
            "player_data_team"
        ],
        team_alias[
            "standings_team"
        ],
    )
)

alias_map.update({
    "Gladbach": "M'gladbach",
    "Luton Town": "Luton",
})

snapshot[
    "old_team_stats_name"
] = (
    snapshot["team"]
    .replace(alias_map)
)

TEAM_STRENGTH_COLS = [
    "team_rank_pct",
    "team_points_per_game",
    "team_goal_diff_per_game",
]

team_strength = (
    team_stats[
        [
            "league",
            "season",
            "team_name",
        ]
        + TEAM_STRENGTH_COLS
    ]
    .drop_duplicates(
        subset=[
            "league",
            "season",
            "team_name",
        ]
    )
    .copy()
)

team_strength[
    "league"
] = (
    team_strength[
        "league"
    ]
    .map(
        canonicalize_league
    )
)

old_strength = (
    team_strength
    .rename(
        columns={
            "team_name": "old_team_stats_name",
            "team_rank_pct": "old_team_rank_pct",
            "team_points_per_game": "old_team_points_per_game",
            "team_goal_diff_per_game": "old_team_goal_diff_per_game",
        }
    )
)

snapshot = snapshot.merge(
    old_strength,
    on=[
        "league",
        "season",
        "old_team_stats_name",
    ],
    how="left",
    validate="many_to_one",
)

old_team_match_rate = (
    snapshot[
        [
            "old_team_rank_pct",
            "old_team_points_per_game",
            "old_team_goal_diff_per_game",
        ]
    ]
    .notna()
    .all(axis=1)
    .mean()
)

print(
    "Old team strength match:",
    f"{old_team_match_rate:.2%}"
)

assert (
    old_team_match_rate
    >= MIN_OLD_TEAM_MATCH_RATE
), (
    "기존 팀 strength 매칭률이 낮습니다. "
    "team alias를 확인하세요."
)

Old team strength match: 100.00%


## 26. v2 — 새 팀의 직전 시즌 T 성적 연결

v1의 coverage가 낮았던 원인 중 하나는:

- `Laliga` vs `La Liga`
- `Borussia Dortmund` vs `Dortmund`
- `Atalanta BC` vs `Atalanta`
- `Manchester United` vs `Man United`

같은 표기 문제였습니다.

v2에서는:

1. league canonicalization
2. 기존 `team_name_alias_map`
3. generic club token 제거
4. club similarity
5. 같은 league 내 best/second-best margin

을 함께 사용합니다.

여전히 직전 시즌 Big5에 없었던 승격팀/외부리그 팀은
정상적인 missing으로 남깁니다.

In [28]:
team_strength_lookup = (
    team_strength
    .copy()
)

team_strength_lookup[
    "norm_team_name"
] = (
    team_strength_lookup[
        "team_name"
    ]
    .map(
        normalize_club
    )
)

# 기존 player_data_team ↔ standings_team alias도
# destination matching 후보로 사용
standings_alias_records = []

for row in team_alias.itertuples(
    index=False
):
    standings_alias_records.append({
        "player_data_team": row.player_data_team,
        "standings_team": row.standings_team,
        "norm_player_team": normalize_club(
            row.player_data_team
        ),
        "norm_standings_team": normalize_club(
            row.standings_team
        ),
    })

standings_alias_df = pd.DataFrame(
    standings_alias_records
)

team_strength_groups = {
    key: group.reset_index(
        drop=True
    )
    for key, group
    in team_strength_lookup.groupby(
        [
            "season",
            "league",
        ]
    )
}


def destination_candidate_score(
    destination_team,
    candidate_team,
):
    score = club_similarity(
        destination_team,
        candidate_team,
    )

    # transfermarkt 이름이 기존 player_data_team alias와
    # 더 잘 맞는 경우 해당 standings 이름 점수도 활용
    alias_rows = (
        standings_alias_df.loc[
            standings_alias_df[
                "standings_team"
            ].eq(
                candidate_team
            )
        ]
    )

    for alias_row in alias_rows.itertuples():
        score = max(
            score,
            club_similarity(
                destination_team,
                alias_row.player_data_team,
            ),
        )

    return float(score)


def match_destination_team_stats(
    season,
    destination_team,
    destination_league,
):
    if (
        pd.isna(destination_team)
        or pd.isna(destination_league)
    ):
        return {
            "new_team_stats_name": np.nan,
            "new_team_match_score": np.nan,
            "new_team_second_score": np.nan,
            "new_team_match_margin": np.nan,
            "new_team_match_method": "not_big5_or_unknown",
        }

    destination_league = (
        canonicalize_league(
            destination_league
        )
    )

    if (
        destination_league
        not in BIG5_LEAGUES
    ):
        return {
            "new_team_stats_name": np.nan,
            "new_team_match_score": np.nan,
            "new_team_second_score": np.nan,
            "new_team_match_margin": np.nan,
            "new_team_match_method": "not_big5_or_unknown",
        }

    destination_team = (
        MANUAL_DESTINATION_TEAM_OVERRIDES
        .get(
            destination_team,
            destination_team,
        )
    )

    key = (
        str(season),
        str(destination_league),
    )

    candidates = (
        team_strength_groups.get(
            key
        )
    )

    if (
        candidates is None
        or candidates.empty
    ):
        return {
            "new_team_stats_name": np.nan,
            "new_team_match_score": np.nan,
            "new_team_second_score": np.nan,
            "new_team_match_margin": np.nan,
            "new_team_match_method": "no_team_stats_candidates",
        }

    scored = candidates[
        [
            "team_name",
        ]
    ].copy()

    scored[
        "score"
    ] = scored[
        "team_name"
    ].map(
        lambda candidate:
            destination_candidate_score(
                destination_team,
                candidate,
            )
    )

    scored = scored.sort_values(
        "score",
        ascending=False,
    ).reset_index(
        drop=True
    )

    best = scored.iloc[0]

    second_score = (
        float(
            scored.iloc[1][
                "score"
            ]
        )
        if len(scored) > 1
        else 0.0
    )

    best_score = float(
        best["score"]
    )

    margin = (
        best_score
        - second_score
    )

    if (
        best_score
        >= NEW_TEAM_MIN_SCORE
        and (
            best_score >= 0.88
            or margin
            >= NEW_TEAM_MIN_MARGIN
        )
    ):
        method = (
            "exact_or_alias"
            if best_score >= 0.98
            else "scored_team_match"
        )

        return {
            "new_team_stats_name": best[
                "team_name"
            ],
            "new_team_match_score": best_score,
            "new_team_second_score": second_score,
            "new_team_match_margin": margin,
            "new_team_match_method": method,
        }

    return {
        "new_team_stats_name": np.nan,
        "new_team_match_score": best_score,
        "new_team_second_score": second_score,
        "new_team_match_margin": margin,
        "new_team_match_method": "unresolved_team_name",
    }


new_team_match_rows = []
cache = {}

for row in snapshot[
    [
        "row_id",
        "season",
        "team",
        "destination_team",
        "destination_league",
        "changed_team_preseason",
        "old_team_stats_name",
    ]
].itertuples(
    index=False
):
    if (
        row.changed_team_preseason
        == 0
    ):
        result = {
            "new_team_stats_name": row.old_team_stats_name,
            "new_team_match_score": 1.0,
            "new_team_second_score": np.nan,
            "new_team_match_margin": np.nan,
            "new_team_match_method": "same_team_copy_old",
        }

    else:
        cache_key = (
            row.season,
            row.destination_team,
            canonicalize_league(
                row.destination_league
            ),
        )

        if (
            cache_key
            not in cache
        ):
            cache[
                cache_key
            ] = (
                match_destination_team_stats(
                    season=row.season,
                    destination_team=row.destination_team,
                    destination_league=row.destination_league,
                )
            )

        result = cache[
            cache_key
        ]

    new_team_match_rows.append({
        "row_id": row.row_id,
        **result,
    })


new_team_matches = (
    pd.DataFrame(
        new_team_match_rows
    )
)

snapshot = snapshot.merge(
    new_team_matches,
    on="row_id",
    how="left",
    validate="one_to_one",
)

new_strength = (
    team_strength
    .rename(
        columns={
            "league": "new_team_strength_league",
            "team_name": "new_team_stats_name",
            "team_rank_pct": "new_team_prev_rank_pct",
            "team_points_per_game": "new_team_prev_points_per_game",
            "team_goal_diff_per_game": "new_team_prev_goal_diff_per_game",
        }
    )
)

snapshot[
    "new_team_strength_league"
] = (
    snapshot[
        "destination_league"
    ]
    .map(
        canonicalize_league
    )
)

snapshot = snapshot.merge(
    new_strength,
    on=[
        "new_team_strength_league",
        "season",
        "new_team_stats_name",
    ],
    how="left",
    validate="many_to_one",
)

snapshot[
    "new_team_strength_missing"
] = (
    snapshot[
        [
            "new_team_prev_rank_pct",
            "new_team_prev_points_per_game",
            "new_team_prev_goal_diff_per_game",
        ]
    ]
    .isna()
    .any(axis=1)
    .astype(int)
)

snapshot[
    "team_rank_change"
] = (
    snapshot[
        "new_team_prev_rank_pct"
    ]
    - snapshot[
        "old_team_rank_pct"
    ]
)

snapshot[
    "team_points_change"
] = (
    snapshot[
        "new_team_prev_points_per_game"
    ]
    - snapshot[
        "old_team_points_per_game"
    ]
)

snapshot[
    "team_goal_diff_change"
] = (
    snapshot[
        "new_team_prev_goal_diff_per_game"
    ]
    - snapshot[
        "old_team_goal_diff_per_game"
    ]
)

print(
    "Changed-team rows:",
    int(
        snapshot[
            "changed_team_preseason"
        ].sum()
    ),
)

changed_big5 = (
    snapshot[
        "changed_team_preseason"
    ].eq(1)
    & snapshot[
        "destination_in_big5"
    ].eq(1)
)

print(
    "Changed team → Big5 rows:",
    int(
        changed_big5.sum()
    ),
)

if changed_big5.any():
    coverage = (
        snapshot.loc[
            changed_big5,
            "new_team_strength_missing",
        ]
        .eq(0)
        .mean()
    )

    print(
        "New-team previous-season strength coverage:",
        f"{coverage:.2%}"
    )

    display(
        snapshot.loc[
            changed_big5,
            [
                "destination_team",
                "destination_league",
                "new_team_stats_name",
                "new_team_match_score",
                "new_team_match_margin",
                "new_team_match_method",
                "new_team_strength_missing",
            ],
        ]
        .head(30)
    )

Changed-team rows: 1071
Changed team → Big5 rows: 707
New-team previous-season strength coverage: 88.97%


,destination_team,destination_league,new_team_stats_name,new_team_match_score,new_team_match_margin,new_team_match_method,new_team_strength_missing
3105,Newcastle United,Premier League,Newcastle,0.920000,0.192727,scored_team_match,0
5152,Bayern Munich,Bundesliga,Bayern Munich,1.000000,0.500000,exact_or_alias,0
5407,Recreativo Huelva,La Liga,NaN,0.400000,0.015385,unresolved_team_name,1
5579,FC Toulouse,Ligue 1,Toulouse,0.980000,0.627059,exact_or_alias,0
6326,Atlético de Madrid,La Liga,Ath Madrid,0.980000,0.359310,exact_or_alias,0
6375,Villarreal CF,La Liga,Villarreal,0.980000,0.480000,exact_or_alias,0
6430,Fulham FC,Premier League,Fulham,0.980000,0.627059,exact_or_alias,0
7073,FC Lorient,Ligue 1,Lorient,0.980000,0.551429,exact_or_alias,0
7231,Olympique Lyon,Ligue 1,NaN,0.444444,0.063492,unresolved_team_name,1
7554,Real Madrid,La Liga,Real Madrid,1.000000,0.238095,exact_or_alias,0


# Part G. Leakage Audit

## 27. Leakage assertions

이 셀은 단순한 확인용 출력이 아니라
**실제로 잘못된 row가 있으면 Notebook을 중단**합니다.

In [29]:
# 1) model에 붙인 selected transfer 날짜는 반드시 cutoff 이하
bad_transfer = snapshot.loc[
    snapshot[
        "transfer_event_preseason"
    ].eq(1)
    & (
        snapshot[
            "transfer_date_preseason"
        ]
        > snapshot[
            "cutoff_date"
        ]
    )
]

assert bad_transfer.empty, (
    "LEAKAGE: cutoff 이후 transfer가 "
    "preseason feature에 포함되었습니다."
)

# 2) transfer_event_preseason=1이면 날짜가 반드시 존재
bad_missing_transfer_date = (
    snapshot.loc[
        snapshot[
            "transfer_event_preseason"
        ].eq(1)
        & snapshot[
            "transfer_date_preseason"
        ].isna()
    ]
)

assert (
    bad_missing_transfer_date.empty
), (
    "LEAKAGE/QUALITY: 날짜 없는 transfer가 "
    "preseason feature에 포함되었습니다."
)

# 3) destination league는 canonical label만 사용
known_destination_league = (
    snapshot[
        "destination_league"
    ]
    .dropna()
)

bad_big5_spellings = (
    known_destination_league[
        known_destination_league
        .astype(str)
        .str.lower()
        .eq("laliga")
    ]
)

assert (
    bad_big5_spellings.empty
), (
    "QUALITY: Laliga 표기가 canonicalize되지 않았습니다."
)

# 4) historical valuation은 cutoff 이하
bad_valuation = snapshot.loc[
    snapshot[
        "mv_preseason_valuation_date"
    ].notna()
    & (
        snapshot[
            "mv_preseason_valuation_date"
        ]
        > snapshot[
            "cutoff_date"
        ]
    )
]

assert bad_valuation.empty, (
    "LEAKAGE: cutoff 이후 시장가치가 포함되었습니다."
)

# 5) 6m valuation
bad_6m = snapshot.loc[
    snapshot[
        "mv_6m_valuation_date"
    ].notna()
    & (
        snapshot[
            "mv_6m_valuation_date"
        ]
        > (
            snapshot[
                "cutoff_date"
            ]
            - pd.Timedelta(
                days=183
            )
        )
    )
]

assert bad_6m.empty, (
    "LEAKAGE: 6m snapshot 날짜 규칙 위반"
)

# 6) 12m valuation
bad_12m = snapshot.loc[
    snapshot[
        "mv_12m_valuation_date"
    ].notna()
    & (
        snapshot[
            "mv_12m_valuation_date"
        ]
        > (
            snapshot[
                "cutoff_date"
            ]
            - pd.Timedelta(
                days=365
            )
        )
    )
]

assert bad_12m.empty, (
    "LEAKAGE: 12m snapshot 날짜 규칙 위반"
)

# 7) games 기반 cutoff는 첫 Big5 경기 이전
games_cutoff_rows = cutoff_table[
    cutoff_table[
        "cutoff_source"
    ].eq(
        "duckdb_big5_first_game"
    )
]

assert (
    games_cutoff_rows[
        "cutoff_date"
    ]
    < games_cutoff_rows[
        "first_big5_game_date"
    ]
).all()

# 8) Final Test input season은 개발 snapshot에 없음
assert (
    "2024-2025"
    not in set(
        snapshot[
            "season"
        ]
        .astype(str)
    )
), (
    "Final Test input season 2024-2025가 "
    "개발 snapshot에 들어왔습니다."
)

print(
    "✅ v2 Leakage / canonicalization assertions passed."
)

✅ v2 Leakage / canonicalization assertions passed.


## 28. Feature construction과 Label 분리 확인

`matched_next`, `next_goals`, `next_10plus`는 최종 데이터에 보존하지만
외부 feature 생성 로직에는 사용하지 않았습니다.

다음 Notebook에서 목적에 따라:

```text
Model A:
전체 row + matched_next

Model B:
matched_next=True + next_goals
```

로 나눌 수 있습니다.

In [30]:
LABEL_COLS = [
    col
    for col in [
        "matched_next",
        "next_goals",
        "next_10plus",
    ]
    if col in snapshot.columns
]

print("Labels preserved:", LABEL_COLS)

Labels preserved: ['matched_next', 'next_goals', 'next_10plus']


# Part H. Coverage / Era Analysis

## 29. 시장가치 시대별 Coverage

사전 검사에서 2000~2004는 시장가치 coverage가 매우 낮았습니다.

따라서 08-02에서는:

### 2000+ 실험
- Transfer state
- New team environment

### 2005+ 실험
- 위 정보
- Market value
- Market momentum

을 별도로 비교할 예정입니다.

여기서는 실제 snapshot coverage를 다시 계산합니다.

In [31]:
coverage_by_target = (
    snapshot
    .groupby(
        "target_season"
    )
    .agg(
        n=("row_id", "size"),
        player_id_coverage=(
            "tm_player_id",
            lambda s:
                s.notna().mean(),
        ),
        market_value_coverage=(
            "market_value_known",
            "mean",
        ),
        confirmed_preseason_transfer_rate=(
            "transfer_event_preseason",
            "mean",
        ),
        final_team_change_rate=(
            "changed_team_preseason",
            "mean",
        ),
        mean_confirmed_event_count=(
            "confirmed_preseason_event_count",
            "mean",
        ),
    )
)

display(
    coverage_by_target
)

market_2005plus_mask = (
    snapshot[
        "target_year"
    ] >= 2005
)

print(
    "Market value coverage all years:",
    f"{snapshot['market_value_known'].mean():.2%}"
)

print(
    "Market value coverage target_year >= 2005:",
    f"{snapshot.loc[market_2005plus_mask, 'market_value_known'].mean():.2%}"
)

print(
    "\nTransfer date coverage by year "
    "(data quality audit; not a model feature):"
)

display(
    transfer_date_coverage_by_year
)

,n,player_id_coverage,market_value_coverage,confirmed_preseason_transfer_rate,final_team_change_rate,mean_confirmed_event_count
target_season,,,,,,
2001-2002,893,0.842105,0.000000,0.000000,0.000000,0.000000
2002-2003,889,0.852643,0.000000,0.000000,0.000000,0.000000
2003-2004,911,0.882547,0.000000,0.000000,0.000000,0.000000
2004-2005,937,0.88047,0.000000,0.001067,0.001067,0.001067
2005-2006,958,0.888309,0.316284,0.000000,0.000000,0.000000
2006-2007,953,0.903463,0.397692,0.002099,0.002099,0.002099
2007-2008,964,0.912863,0.466805,0.004149,0.004149,0.004149
2008-2009,1010,0.924752,0.551485,0.001980,0.001980,0.001980
2009-2010,984,0.920732,0.621951,0.004065,0.004065,0.004065


Market value coverage all years: 68.16%
Market value coverage target_year >= 2005: 80.71%

Transfer date coverage by year (data quality audit; not a model feature):


,season,events,dated_events,date_coverage
0,2000,1460,0,0.000000
1,2001,1412,0,0.000000
2,2002,1495,1,0.000669
3,2003,1575,2,0.001270
4,2004,1747,6,0.003434
5,2005,1790,13,0.007263
6,2006,1747,21,0.012021
7,2007,1818,46,0.025303
8,2008,1801,51,0.028318
9,2009,1821,68,0.037342


## 30. matched_next별 이적 비율 — 설명용 Audit

이 값은 feature 생성에 사용하지 않습니다.

다만 다음 단계 Model A에서
이적 정보가 Big5 잔류 여부와 얼마나 관계가 있는지 파악하기 위한 사전 통계입니다.

In [32]:
display(
    snapshot
    .groupby(
        "matched_next"
    )
    .agg(
        n=("row_id", "size"),
        confirmed_preseason_transfer_rate=(
            "transfer_event_preseason",
            "mean",
        ),
        final_team_change_rate=(
            "changed_team_preseason",
            "mean",
        ),
        destination_big5_rate=(
            "destination_in_big5",
            "mean",
        ),
        market_value_coverage=(
            "market_value_known",
            "mean",
        ),
    )
)

print(
    "\n주의: matched_next는 label이므로 "
    "이 표는 설명용 audit이며 feature 생성에 사용하지 않습니다."
)

,n,confirmed_preseason_transfer_rate,final_team_change_rate,destination_big5_rate,market_value_coverage
matched_next,,,,,
False,3950,0.046582,0.043797,0.956456,0.566835
True,19403,0.056847,0.046282,0.990105,0.704994



주의: matched_next는 label이므로 이 표는 설명용 audit이며 feature 생성에 사용하지 않습니다.


# Part I. 기존 Error Case 검증

## 31. Retegui / Greenwood / Guirassy / Dembélé 확인

08을 시작하게 만든 실제 오류 사례가
새 데이터에서 어떻게 표현되는지 직접 확인합니다.

기대:

- Retegui: Genoa → Atalanta 이적
- Greenwood: 다음 시즌 새 팀 이동
- Guirassy: Stuttgart → Dortmund 이적
- Dembélé: 해당 여름 큰 팀 이동 없이 market value 정보 중심

이 셀은 단순 예시가 아니라
entity resolution과 transfer event가 상식적으로 맞는지 확인하는 sanity check입니다.

In [33]:
CASE_PLAYERS = [
    "Mateo Retegui",
    "Mason Greenwood",
    "Serhou Guirassy",
    "Ousmane Dembélé",
]

case_rows = snapshot.loc[
    snapshot[
        "player"
    ].isin(
        CASE_PLAYERS
    ),
    [
        "player",
        "season",
        "target_season",
        "team",
        "tm_player_id",
        "match_confidence",
        "cutoff_date",

        "timeline_status",
        "confirmed_preseason_event_count",
        "preseason_team_transition_count",
        "timeline_trace",

        "transfer_event_preseason",
        "changed_team_preseason",
        "destination_team",
        "destination_league",
        "destination_in_big5",
        "transfer_date_preseason",
        "transfer_fee_eur",

        "market_value_preseason_eur",
        "market_value_growth_6m",
        "market_value_growth_12m",

        "old_team_rank_pct",
        "new_team_prev_rank_pct",
        "team_rank_change",
        "new_team_match_method",

        "next_goals",
    ],
].sort_values(
    [
        "player",
        "season",
    ]
)

display(
    case_rows
)

print(
    "\nExpected sanity checks:"
)
print(
    "- Retegui 2023-24 → Atalanta: changed_team_preseason=1"
)
print(
    "- Guirassy 2022-23 → 2023-24: final destination should remain Stuttgart"
)
print(
    "- Guirassy 2023-24 → Dortmund: changed_team_preseason=1"
)
print(
    "- Greenwood 2023-24: if Marseille date is successfully 1:1 matched "
    "and before cutoff, destination should become Marseille"
)

,player,season,target_season,team,tm_player_id,match_confidence,cutoff_date,timeline_status,confirmed_preseason_event_count,preseason_team_transition_count,...,transfer_date_preseason,transfer_fee_eur,market_value_preseason_eur,market_value_growth_6m,market_value_growth_12m,old_team_rank_pct,new_team_prev_rank_pct,team_rank_change,new_team_match_method,next_goals
19251,Mason Greenwood,2019-2020,2020-2021,Manchester Utd,532826,A,2020-08-20,no_summer_event,0,0,...,NaT,NaN,45000000.0,1.250000,5.428571,0.894737,0.894737,0.000000,same_team_copy_old,7.0
20179,Mason Greenwood,2020-2021,2021-2022,Manchester Utd,532826,A,2021-08-05,no_summer_event,0,0,...,NaT,NaN,50000000.0,0.000000,0.111111,0.947368,0.947368,0.000000,same_team_copy_old,5.0
21129,Mason Greenwood,2021-2022,2022-2023,Manchester Utd,532826,A,2022-08-04,no_summer_event,0,0,...,NaT,NaN,50000000.0,0.000000,0.000000,0.736842,0.736842,0.000000,same_team_copy_old,0.0
23008,Mason Greenwood,2023-2024,2024-2025,Getafe,532826,A,2024-08-14,confirmed_pre_cutoff_event,2,2,...,2024-07-18,26000000.0,25000000.0,2.333333,-0.500000,0.421053,0.588235,0.167183,scored_team_match,21.0
23010,Mateo Retegui,2023-2024,2024-2025,Genoa,554903,A,2024-08-14,confirmed_pre_cutoff_event,1,1,...,2024-08-08,20900000.0,16000000.0,0.000000,0.000000,0.473684,0.842105,0.368421,exact_or_alias,25.0
15538,Ousmane Dembélé,2015-2016,2016-2017,Rennes,288230,A,2016-08-11,no_confirmed_pre_cutoff_event,0,0,...,NaT,NaN,14000000.0,3.666667,NaN,0.631579,0.631579,0.000000,same_team_copy_old,6.0
16515,Ousmane Dembélé,2016-2017,2017-2018,Dortmund,288230,A,2017-08-03,no_confirmed_pre_cutoff_event,0,0,...,NaT,NaN,33000000.0,0.833333,1.357143,0.882353,0.882353,0.000000,same_team_copy_old,3.0
17483,Ousmane Dembélé,2017-2018,2018-2019,Barcelona,288230,A,2018-08-09,no_summer_event,0,0,...,NaT,NaN,80000000.0,0.000000,1.424242,1.000000,1.000000,0.000000,same_team_copy_old,8.0
18434,Ousmane Dembélé,2018-2019,2019-2020,Barcelona,288230,A,2019-08-08,no_summer_event,0,0,...,NaT,NaN,100000000.0,0.250000,0.250000,1.000000,1.000000,0.000000,same_team_copy_old,1.0
20297,Ousmane Dembélé,2020-2021,2021-2022,Barcelona,288230,A,2021-08-05,no_summer_event,0,0,...,NaT,NaN,50000000.0,0.000000,-0.107143,0.894737,0.894737,0.000000,same_team_copy_old,1.0



Expected sanity checks:
- Retegui 2023-24 → Atalanta: changed_team_preseason=1
- Guirassy 2022-23 → 2023-24: final destination should remain Stuttgart
- Guirassy 2023-24 → Dortmund: changed_team_preseason=1
- Greenwood 2023-24: if Marseille date is successfully 1:1 matched and before cutoff, destination should become Marseille


# Part J. Data Quality Summary

## 32. 핵심 품질 지표

In [34]:
changed_big5_mask = (
    snapshot[
        "changed_team_preseason"
    ].eq(1)
    & snapshot[
        "destination_in_big5"
    ].eq(1)
)

quality_metrics = [
    {
        "metric": "development_rows",
        "value": len(snapshot),
    },
    {
        "metric": "unique_players",
        "value": snapshot[
            "player"
        ].nunique(),
    },
    {
        "metric": "player_id_unique_match_rate",
        "value": player_unique_match_rate,
    },
    {
        "metric": "player_id_row_match_rate",
        "value": player_row_match_rate,
    },
    {
        "metric": "old_team_strength_match_rate",
        "value": old_team_match_rate,
    },
    {
        "metric": "summer_event_transfer_date_coverage",
        "value": transfer_date_match_rate,
    },
    {
        "metric": "confirmed_preseason_transfer_rate",
        "value": snapshot[
            "transfer_event_preseason"
        ].mean(),
    },
    {
        "metric": "final_changed_team_rate",
        "value": snapshot[
            "changed_team_preseason"
        ].mean(),
    },
    {
        "metric": "market_value_coverage_all",
        "value": snapshot[
            "market_value_known"
        ].mean(),
    },
    {
        "metric": "market_value_coverage_2005plus",
        "value": snapshot.loc[
            market_2005plus_mask,
            "market_value_known",
        ].mean(),
    },
    {
        "metric": "post_cutoff_context_events_audit",
        "value": len(
            post_cutoff_transfer_audit
        ),
    },
    {
        "metric": "unknown_date_context_events_audit",
        "value": len(
            unknown_date_transfer_audit
        ),
    },
    {
        "metric": "league_labels_normalized",
        "value": int(
            league_normalization_changes
        ),
    },
]

if changed_big5_mask.any():
    quality_metrics.append({
        "metric": "new_team_strength_coverage_changed_big5",
        "value": snapshot.loc[
            changed_big5_mask,
            "new_team_strength_missing",
        ].eq(0).mean(),
    })

data_quality_summary = (
    pd.DataFrame(
        quality_metrics
    )
)

data_quality_summary

,metric,value
0,development_rows,23353.000000
1,unique_players,6161.000000
2,player_id_unique_match_rate,0.901152
3,player_id_row_match_rate,0.925106
4,old_team_strength_match_rate,1.000000
5,summer_event_transfer_date_coverage,0.278556
6,confirmed_preseason_transfer_rate,0.055111
7,final_changed_team_rate,0.045861
8,market_value_coverage_all,0.681625
9,market_value_coverage_2005plus,0.807078


## 33. 외부 Feature 그룹 정리

08-02에서 한꺼번에 넣지 않고 그룹 단위 ablation을 할 수 있도록
column 목록을 미리 고정합니다.

In [35]:
TRANSFER_STATE_FEATURES = [
    # confirmed dated information only
    "transfer_event_preseason",
    "changed_team_preseason",
    "same_league_transfer",
    "league_changed",
    "country_changed",
    "destination_in_big5",
    "is_loan_preseason",
    "days_since_transfer",

    # 여러 event가 있는 경우 추가 정보
    "confirmed_preseason_event_count",
    "preseason_team_transition_count",
]

TRANSFER_ECONOMIC_FEATURES = [
    "transfer_fee_known",
    "transfer_fee_positive",
    "log_transfer_fee",
    "fee_to_transfer_market_value_ratio",
]

MARKET_VALUE_FEATURES = [
    "market_value_known",
    "log_market_value",
    "market_value_percentile",
    "market_value_vs_position_median",
    "valuation_age_days",
]

MARKET_MOMENTUM_FEATURES = [
    "market_value_growth_6m",
    "market_value_growth_12m",
    "market_value_vs_peak",
]

NEW_TEAM_ENVIRONMENT_FEATURES = [
    "new_team_prev_rank_pct",
    "new_team_prev_points_per_game",
    "new_team_prev_goal_diff_per_game",
    "team_rank_change",
    "team_points_change",
    "team_goal_diff_change",
    "new_team_strength_missing",
]

EXTERNAL_FEATURE_GROUPS = {
    "transfer_state": (
        TRANSFER_STATE_FEATURES
    ),
    "transfer_economics": (
        TRANSFER_ECONOMIC_FEATURES
    ),
    "market_value": (
        MARKET_VALUE_FEATURES
    ),
    "market_momentum": (
        MARKET_MOMENTUM_FEATURES
    ),
    "new_team_environment": (
        NEW_TEAM_ENVIRONMENT_FEATURES
    ),
}

print(
    "AUDIT ONLY — model feature에 넣지 않을 것:"
)
print(
    "- unknown_date_context_event_count_audit"
)
print(
    "- post_cutoff_context_event_count_audit"
)
print(
    "- timeline_status / timeline_trace"
)
print()

for group, cols in (
    EXTERNAL_FEATURE_GROUPS.items()
):
    print(
        f"{group}: {len(cols)}"
    )

    for col in cols:
        print(
            "  -",
            col,
        )

AUDIT ONLY — model feature에 넣지 않을 것:
- unknown_date_context_event_count_audit
- post_cutoff_context_event_count_audit
- timeline_status / timeline_trace

transfer_state: 10
  - transfer_event_preseason
  - changed_team_preseason
  - same_league_transfer
  - league_changed
  - country_changed
  - destination_in_big5
  - is_loan_preseason
  - days_since_transfer
  - confirmed_preseason_event_count
  - preseason_team_transition_count
transfer_economics: 4
  - transfer_fee_known
  - transfer_fee_positive
  - log_transfer_fee
  - fee_to_transfer_market_value_ratio
market_value: 5
  - market_value_known
  - log_market_value
  - market_value_percentile
  - market_value_vs_position_median
  - valuation_age_days
market_momentum: 3
  - market_value_growth_6m
  - market_value_growth_12m
  - market_value_vs_peak
new_team_environment: 7
  - new_team_prev_rank_pct
  - new_team_prev_points_per_game
  - new_team_prev_goal_diff_per_game
  - team_rank_change
  - team_points_change
  - team_goal_diff_cha

# Part K. 저장

## 34. 중간 산출물 저장

Data pipeline에서는 최종 CSV 하나만 저장하면
나중에 잘못된 join을 찾기 어렵습니다.

따라서 crosswalk / event / audit를 각각 저장합니다.

In [36]:
OUTPUT_PATHS = {
    "cutoff": (
        ARTIFACT_DIR
        / "08_v2_cutoff_table.csv"
    ),
    "player_crosswalk": (
        ARTIFACT_DIR
        / "08_v2_player_crosswalk.csv"
    ),
    "player_season_crosswalk": (
        ARTIFACT_DIR
        / "08_v2_player_season_crosswalk.csv"
    ),
    "player_match_audit": (
        ARTIFACT_DIR
        / "08_v2_player_match_audit.csv"
    ),
    "unmatched_players": (
        ARTIFACT_DIR
        / "08_v2_unmatched_players.csv"
    ),
    "transfer_events": (
        ARTIFACT_DIR
        / "08_v2_transfer_events_clean.csv"
    ),
    "transfer_date_audit": (
        ARTIFACT_DIR
        / "08_v2_transfer_date_match_audit.csv"
    ),
    "transfer_date_coverage": (
        ARTIFACT_DIR
        / "08_v2_transfer_date_coverage_by_year.csv"
    ),
    "post_cutoff_transfer": (
        ARTIFACT_DIR
        / "08_v2_post_cutoff_transfer_audit.csv"
    ),
    "unknown_date_transfer": (
        ARTIFACT_DIR
        / "08_v2_unknown_date_transfer_audit.csv"
    ),
    "market_value_snapshot": (
        ARTIFACT_DIR
        / "08_v2_market_value_snapshots.csv"
    ),
    "quality": (
        ARTIFACT_DIR
        / "08_v2_data_quality_summary.csv"
    ),
    "final_snapshot": (
        ARTIFACT_DIR
        / "08_v2_preseason_player_snapshot_dev.csv"
    ),
}

cutoff_table.to_csv(
    OUTPUT_PATHS["cutoff"],
    index=False,
)

player_crosswalk.to_csv(
    OUTPUT_PATHS[
        "player_crosswalk"
    ],
    index=False,
)

player_season_crosswalk.to_csv(
    OUTPUT_PATHS[
        "player_season_crosswalk"
    ],
    index=False,
)

player_match_audit.to_csv(
    OUTPUT_PATHS[
        "player_match_audit"
    ],
    index=False,
)

unmatched_players.to_csv(
    OUTPUT_PATHS[
        "unmatched_players"
    ],
    index=False,
)

transfer_events.to_csv(
    OUTPUT_PATHS[
        "transfer_events"
    ],
    index=False,
)

transfer_date_matches.to_csv(
    OUTPUT_PATHS[
        "transfer_date_audit"
    ],
    index=False,
)

transfer_date_coverage_by_year.to_csv(
    OUTPUT_PATHS[
        "transfer_date_coverage"
    ],
    index=False,
)

post_cutoff_transfer_audit.to_csv(
    OUTPUT_PATHS[
        "post_cutoff_transfer"
    ],
    index=False,
)

unknown_date_transfer_audit.to_csv(
    OUTPUT_PATHS[
        "unknown_date_transfer"
    ],
    index=False,
)

market_value_snapshots.to_csv(
    OUTPUT_PATHS[
        "market_value_snapshot"
    ],
    index=False,
)

data_quality_summary.to_csv(
    OUTPUT_PATHS[
        "quality"
    ],
    index=False,
)

snapshot.to_csv(
    OUTPUT_PATHS[
        "final_snapshot"
    ],
    index=False,
)

print("Saved v2:")
for name, path in (
    OUTPUT_PATHS.items()
):
    print(
        f"- {name:<24}",
        path.resolve(),
    )

Saved v2:
- cutoff                   D:\dev\03_PersonalProjects\next_season_goal_prediction\notebooks\artifacts\08_v2_cutoff_table.csv
- player_crosswalk         D:\dev\03_PersonalProjects\next_season_goal_prediction\notebooks\artifacts\08_v2_player_crosswalk.csv
- player_season_crosswalk  D:\dev\03_PersonalProjects\next_season_goal_prediction\notebooks\artifacts\08_v2_player_season_crosswalk.csv
- player_match_audit       D:\dev\03_PersonalProjects\next_season_goal_prediction\notebooks\artifacts\08_v2_player_match_audit.csv
- unmatched_players        D:\dev\03_PersonalProjects\next_season_goal_prediction\notebooks\artifacts\08_v2_unmatched_players.csv
- transfer_events          D:\dev\03_PersonalProjects\next_season_goal_prediction\notebooks\artifacts\08_v2_transfer_events_clean.csv
- transfer_date_audit      D:\dev\03_PersonalProjects\next_season_goal_prediction\notebooks\artifacts\08_v2_transfer_date_match_audit.csv
- transfer_date_coverage   D:\dev\03_PersonalProjects\next_season_g

## 35. 최종 snapshot 핵심 컬럼 확인

In [37]:
important_cols = [
    "row_id",
    "player",
    "season",
    "target_season",
    "team",
    "league",
    "tm_player_id",
    "match_confidence",
    "cutoff_date",
    "cutoff_source",

    # transfer timeline
    "timeline_status",
    "confirmed_preseason_event_count",
    "preseason_team_transition_count",
    "transfer_event_preseason",
    "changed_team_preseason",
    "destination_team",
    "destination_league",
    "destination_in_big5",
    "is_loan_preseason",
    "days_since_transfer",
    "transfer_fee_eur",

    # market
    "market_value_preseason_eur",
    "market_value_source",
    "market_value_growth_6m",
    "market_value_growth_12m",
    "market_value_vs_peak",
    "market_value_percentile",

    # environment
    "old_team_rank_pct",
    "new_team_prev_rank_pct",
    "team_rank_change",
    "team_points_change",
    "new_team_strength_missing",
    "new_team_match_method",

    # labels
    "matched_next",
    "next_goals",
]

available_important = [
    c
    for c in important_cols
    if c in snapshot.columns
]

display(
    snapshot[
        available_important
    ].head(20)
)

print(
    "Final v2 snapshot shape:",
    snapshot.shape,
)

,row_id,player,season,target_season,team,league,tm_player_id,match_confidence,cutoff_date,cutoff_source,...,market_value_vs_peak,market_value_percentile,old_team_rank_pct,new_team_prev_rank_pct,team_rank_change,team_points_change,new_team_strength_missing,new_team_match_method,matched_next,next_goals
0,0,Abdelhafid Tasfaout,2000-2001,2001-2002,Guingamp,Ligue 1,155601,A,2001-07-31,fallback_jul31,...,NaN,NaN,0.470588,0.470588,0.0,0.0,0,same_team_copy_old,True,1.0
1,1,Abder Ramdane,2000-2001,2001-2002,Freiburg,Bundesliga,<NA>,UNMATCHED,2001-07-31,fallback_jul31,...,NaN,NaN,0.705882,0.705882,0.0,0.0,0,same_team_copy_old,True,0.0
2,2,Adaílton,2000-2001,2001-2002,Hellas Verona,Serie A,<NA>,UNMATCHED,2001-07-31,fallback_jul31,...,NaN,NaN,0.176471,0.176471,0.0,0.0,0,same_team_copy_old,True,1.0
3,3,Ade Akinbiyi,2000-2001,2001-2002,Leicester City,Premier League,4037,A,2001-07-31,fallback_jul31,...,NaN,NaN,0.368421,0.368421,0.0,0.0,0,same_team_copy_old,True,2.0
4,4,Adel Sellimi,2000-2001,2001-2002,Freiburg,Bundesliga,<NA>,UNMATCHED,2001-07-31,fallback_jul31,...,NaN,NaN,0.705882,0.705882,0.0,0.0,0,same_team_copy_old,True,5.0
5,5,Adrian Mutu,2000-2001,2001-2002,Hellas Verona,Serie A,5879,A,2001-07-31,fallback_jul31,...,NaN,NaN,0.176471,0.176471,0.0,0.0,0,same_team_copy_old,True,12.0
6,6,Adriano Gabiru,2000-2001,2001-2002,Marseille,Ligue 1,20403,A,2001-07-31,fallback_jul31,...,NaN,NaN,0.176471,0.176471,0.0,0.0,0,same_team_copy_old,False,0.0
7,7,Agostinho,2000-2001,2001-2002,Málaga,La Liga,3622,A,2001-07-31,fallback_jul31,...,NaN,NaN,0.631579,0.631579,0.0,0.0,0,same_team_copy_old,True,0.0
8,8,Aílton Gonçalves,2000-2001,2001-2002,Werder Bremen,Bundesliga,<NA>,UNMATCHED,2001-07-31,fallback_jul31,...,NaN,NaN,0.647059,0.647059,0.0,0.0,0,same_team_copy_old,True,16.0
9,9,Aimo Diana,2000-2001,2001-2002,Brescia,Serie A,6749,A,2001-07-31,fallback_jul31,...,NaN,NaN,0.588235,0.588235,0.0,0.0,0,same_team_copy_old,True,1.0


Final v2 snapshot shape: (23353, 114)


# 36. v2 결과 해석 — 직접 작성

아래는 **의도적으로 비워둡니다.**

## v1 → v2에서 반드시 확인할 것

1. Player ID 매칭률은 기존 약 90% / 92.5% 수준을 유지하는가?
2. `Laliga`가 모두 `La Liga`로 정규화되었는가?
3. summer event transfer-date coverage가 v1의 22.83%보다 개선되었는가?
4. Greenwood 2023-24 → 2024-25 Marseille 이동이 cutoff 이전 event로 잡히는가?
5. Guirassy 2022-23 → 2023-24의 최종 destination이 Stuttgart로 유지되는가?
6. Guirassy 2023-24 → 2024-25 Dortmund 이동이 정상적으로 잡히는가?
7. Retegui Genoa → Atalanta가 그대로 정상인가?
8. 새 Big5 팀 strength coverage가 v1의 58.46%보다 개선되는가?
9. cutoff 이후 event가 model feature에 들어오지 않았는가?
10. 날짜 미확인 summer event가 model feature에 사용되지 않았는가?
11. market value coverage 68.16% / 2005+ 80.71% 수준이 유지되는가?
12. leakage assertions가 모두 통과하는가?

---

## 내 결론

- Player ID 품질:
- Transfer date coverage:
- Transfer timeline 품질:
- League canonicalization:
- New-team strength coverage:
- Market value 품질:
- Leakage check:
- 08-02 진행 가능 여부:
- 추가 보정 필요 항목:

# 다음 단계: 08-02 External Feature Ablation

v2 품질 검사가 통과하면 모델링으로 넘어갑니다.

## 중요한 원칙

- 08-02에서 다시 Optuna부터 돌리지 않음
- 먼저 **고정 Baseline CatBoost**로 외부 데이터 그룹 자체의 가치를 측정
- transfer-date coverage가 시대별로 다르므로 전체 기간 결과와 최근 기간 sensitivity를 함께 확인
- unknown-date / post-cutoff audit 정보는 model feature로 절대 사용하지 않음

### 기본 실험

```text
Exp A
Exp9Cb

Exp B
+ confirmed Transfer State

Exp C
+ Transfer Economics

Exp D
+ Market Value

Exp E
+ Market Momentum

Exp F
+ New Team Environment
```

시장가치 실험은 동일한 2005+ subset끼리 비교합니다.

Transfer feature는 `08_v2_transfer_date_coverage_by_year.csv`를 보고
최근 시대 sensitivity analysis 범위를 먼저 결정합니다.